# Polymer Property Prediction — Round 3

Seven targets (`tg, egc, egb, eps, nc, ei, eea`), scored as the unweighted mean R² across them.

### What decides the score

Five of the seven properties have **fewer than 350 training rows** (egb 337, eps 229, nc 229,
ei 222, eea 221) while `tg` has 4,143. Because the metric averages R² over targets without
weighting, **71% of the score is decided by properties with under 350 labels each**. Every
design choice below is aimed at the small five.

### The three levers

1. **Polymer invariance.** `poly_key = ring_key(reduce_repeat(smiles))` is exactly invariant to
   both Round-3 invariances — SMILES translation *and* monomer/dimer/trimer repetition. It is
   verified on real data in section 1 (the notebook aborts if invariance fails). It is used to
   normalise every molecule before featurisation, to key the partner join, and to group the CV
   folds so a molecule cannot sit in train and validation at once.

2. **PI1M transfer.** `PI1M.csv` holds 995,799 polymer SMILES, 100% carrying `*` end markers,
   mean length 46.8 against train's 49.3 — the same chemical domain. A transformer encoder is
   pretrained on it from scratch with masked-language modelling, then fine-tuned per property.
   This is the main new lever for the data-poor five. (`smile_r3.csv` is 5.97M drug-like
   molecules with **0%** `*` markers — a different domain, so it is used only as optional extra
   MLM text, never as the primary corpus.)

3. **Co-observation and physics.** The six DFT properties are co-observed on the same molecules
   in train and test alike, so a partner's *true* value is a legitimate feature. Keying that join
   on `poly_key` instead of the raw string lifts mean test partner coverage from 0.391 to 0.468.
   The near-exact identities `ei = egc + eea`, `egb ≈ egc`, `eps ≈ nc²` route the abundant `egc`
   labels into the starved `ei`/`eea`/`egb` targets.

### Leakage control

`true_egc` **is the answer** when the target is `egc`. Every engineered column declares the
labels it was built from in `USES`, and two guards apply: `drop_leaky()` for per-property models,
`mask_rows_for_multitask()` for the shared-trunk models. Section 6 proves the leak exists, then
proves each guard removes it, and aborts if either check fails.

### Compliance

Only `train.csv`, `test.csv`, `PI1M.csv` and `smile_r3.csv` from the official bundle are read.
No pretrained weights are downloaded — the encoder is trained from scratch inside this notebook.
No test label is ever read or inferred. Every artifact is produced by this notebook's own run.

### Resumability

Every stage writes an atomically-replaced artifact and a manifest entry. Re-running the notebook
after any failure discovers what already completed and resumes at the first incomplete stage.
Gradient-boosting resumes per fold; neural training resumes **per epoch**, restoring optimiser
and scheduler state.

In [ ]:
import subprocess, sys

for _pkg in ['rdkit', 'shap']:
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=False)
print('deps ready')

In [ ]:
import os, sys, time, json, pickle, warnings, gc, re, glob, hashlib, traceback, math, random
from datetime import datetime
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem, RDLogger
from rdkit.Chem import (Descriptors, AllChem, MACCSkeys, rdMolDescriptors,
                        Lipinski, rdFingerprintGenerator)
RDLogger.logger().setLevel(RDLogger.CRITICAL)

HAVE_XGB = HAVE_CB = HAVE_SHAP = False
try:
    import xgboost as xgb; HAVE_XGB = True
except ImportError:
    pass
try:
    import catboost as cb; HAVE_CB = True
except ImportError:
    pass
try:
    import shap; HAVE_SHAP = True
except ImportError:
    pass

SEED = 42
N_FOLDS = 10
TARGET_TYPES = ['tg', 'egc', 'egb', 'eps', 'nc', 'ei', 'eea']
DFT_PROPS = ['egc', 'egb', 'ei', 'eea', 'eps', 'nc']       # co-observed block; tg is disjoint
TASK_MAP = {t: i for i, t in enumerate(TARGET_TYPES)}
MORGAN_BITS_R2, MORGAN_BITS_R3, AP_BITS, TT_BITS = 2048, 1024, 1024, 1024

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- which stages run -------------------------------------------------------------------
# 'auto' enables the PI1M encoder, its fine-tuning and the CNN only when a GPU is present.
# Force with PPP_NEURAL=0 to get a CPU-only run: gradient boosting + multi-task NN + partner
# features + physics + stacking still produce a complete, valid submission. Useful for a free
# Kaggle CPU session that validates the backbone without spending GPU quota.
_NEURAL = os.environ.get('PPP_NEURAL', 'auto').lower()
USE_NEURAL = torch.cuda.is_available() if _NEURAL == 'auto' else _NEURAL in ('1', 'true', 'yes')
NN_SEED_COUNT = int(os.environ.get('PPP_NN_SEEDS', '5'))

# Stages to skip this run, comma separated, e.g. PPP_SKIP=12_finetune,13_cnn
# A skipped stage returns None and writes no checkpoint, so a later run can still produce it.
# An already-completed stage is loaded rather than skipped - finished work is never discarded.
# Not part of the run signature, so skipping does not invalidate an existing checkpoint store.
SKIP_STAGES = {x.strip() for x in os.environ.get('PPP_SKIP', '').split(',') if x.strip()}

# ---- wall-clock budget (seconds). Stages that can be shortened honour their own cap. ----
T_START = time.time()
BUDGET = dict(total=11.0 * 3600, pretrain=70 * 60, finetune=110 * 60, cnn=35 * 60, nn=35 * 60)


def elapsed():
    return time.time() - T_START


def time_left():
    return BUDGET['total'] - elapsed()

## 0. Auto-discovery, logging and checkpointing

Nothing below is configured by hand. The data root is found by searching for a directory that
contains both `train.csv` and `test.csv`; the checkpoint store is scanned at startup and every
completed stage is reported before any work begins.

In [ ]:
def _discover_data_dir():
    """Find the directory holding train.csv and test.csv, wherever the notebook is run."""
    named = ['/kaggle/input/competitions/ppp-round-3', '/kaggle/input/ppp-round-3',
             '/kaggle/input/aisehack-2-0-polymer-property-prediction-round-3',
             './ppp-round-3', '.', './input/ppp-round-3']
    for p in named:
        if os.path.exists(os.path.join(p, 'train.csv')) and os.path.exists(os.path.join(p, 'test.csv')):
            return os.path.abspath(p)
    for root in ['/kaggle/input', os.getcwd(), os.path.dirname(os.getcwd())]:
        if not os.path.isdir(root):
            continue
        for hit in glob.glob(os.path.join(root, '**', 'train.csv'), recursive=True):
            d = os.path.dirname(hit)
            if os.path.exists(os.path.join(d, 'test.csv')):
                return d
    raise FileNotFoundError('could not locate train.csv + test.csv')


DATA_DIR = _discover_data_dir()
WORK_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
CKPT_DIR = os.path.join(WORK_DIR, 'ckpt')
FOLD_DIR = os.path.join(CKPT_DIR, 'folds')
os.makedirs(FOLD_DIR, exist_ok=True)


def _find_aux(*names):
    """Locate an optional auxiliary csv by name, searching the data root then the whole input tree."""
    for n in names:
        p = os.path.join(DATA_DIR, n)
        if os.path.exists(p):
            return p
    for root in ['/kaggle/input', os.path.dirname(DATA_DIR), os.getcwd()]:
        if not os.path.isdir(root):
            continue
        for n in names:
            hits = glob.glob(os.path.join(root, '**', n), recursive=True)
            if hits:
                return hits[0]
    return None


PI1M_PATH = _find_aux('PI1M.csv', 'pi1m.csv')
AUX_PATH = _find_aux('smile_r3.csv', 'smiles_r3.csv')


class Logger:
    """Prints to stdout and mirrors to run.log plus a machine-readable events.jsonl."""

    LEVELS = ('DEBUG', 'INFO', 'METRIC', 'OK', 'WARN', 'ERROR')

    def __init__(self, work_dir):
        self.txt = os.path.join(work_dir, 'run.log')
        self.jsonl = os.path.join(work_dir, 'events.jsonl')
        self.stage = '-'
        self._say('INFO', f'=== logger started {datetime.now():%Y-%m-%d %H:%M:%S} ===')

    def _say(self, lvl, msg, **kv):
        # Deliberately narrow: time, level, message. Elapsed time and the current stage go to
        # events.jsonl for machine reading rather than into every console line, where they push
        # the message 30 characters right and make the log hard to scan.
        line = f'[{datetime.now():%H:%M:%S}] [{lvl:>6}] {msg}'
        print(line, flush=True)
        try:
            with open(self.txt, 'a') as h:
                h.write(line + '\n')
            with open(self.jsonl, 'a') as h:
                h.write(json.dumps({'ts': datetime.now().isoformat(), 'elapsed_s': round(elapsed(), 1),
                                    'level': lvl, 'stage': self.stage, 'msg': msg, **kv}) + '\n')
        except Exception:
            pass

    def debug(self, m, **kv): self._say('DEBUG', m, **kv)
    def info(self, m, **kv): self._say('INFO', m, **kv)
    def metric(self, m, **kv): self._say('METRIC', m, **kv)
    def ok(self, m, **kv): self._say('OK', m, **kv)
    def warn(self, m, **kv): self._say('WARN', m, **kv)
    def error(self, m, **kv): self._say('ERROR', m, **kv)

    def header(self, m):
        self._say('INFO', '=' * 60)
        self._say('INFO', f'  {m}')
        self._say('INFO', '=' * 60)

    def sub(self, m): self._say('INFO', f'--- {m} ---')

    def set_stage(self, s): self.stage = s


log = Logger(WORK_DIR)


def _atomic_write(path, writer):
    """Write via a temp file then rename, so an interrupt can never leave a half-written artifact."""
    tmp = f'{path}.tmp{os.getpid()}'
    try:
        writer(tmp)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            try:
                os.remove(tmp)
            except OSError:
                pass


class Ckpt:
    """Stage store. Every entry records status, an input key, the artifact path and timings."""

    def __init__(self, root):
        self.root = root
        self.mpath = os.path.join(root, 'manifest.json')
        self.man = {}
        if os.path.exists(self.mpath):
            try:
                self.man = json.load(open(self.mpath))
            except Exception:
                log.warn('manifest unreadable - starting a fresh one')
                self.man = {}

    def _flush(self):
        _atomic_write(self.mpath,
                      lambda p: json.dump(self.man, open(p, 'w'), indent=1, default=str))

    def path(self, name):
        return os.path.join(self.root, f'{name.replace("/", "__")}.pkl')

    def resolve(self, name):
        """Locate an artifact by basename under the current root.

        The manifest records an absolute path, but a checkpoint set imported from a previous
        Kaggle run arrives under /kaggle/input, not the path it was written to. Resolving by
        basename makes the store portable between sessions.
        """
        e = self.man.get(name) or {}
        f = e.get('file', '')
        local = os.path.join(self.root, os.path.basename(f)) if f else self.path(name)
        if os.path.exists(local):
            return local
        return f if f and os.path.exists(f) else None

    def has(self, name, key=''):
        e = self.man.get(name)
        return bool(e and e.get('status') == 'done' and e.get('key') == key
                    and self.resolve(name))

    def load(self, name):
        with open(self.resolve(name), 'rb') as h:
            return pickle.load(h)

    def put(self, name, obj, key='', meta=None):
        f = self.path(name)
        _atomic_write(f, lambda p: pickle.dump(obj, open(p, 'wb'), protocol=4))
        self.man[name] = dict(status='done', key=key, file=f, meta=meta or {},
                              ts=datetime.now().isoformat())
        self._flush()

    def fail(self, name, err):
        self.man[name] = dict(status='failed', error=str(err)[-3000:],
                              ts=datetime.now().isoformat())
        self._flush()

    def summary(self):
        items = [(k, v) for k, v in self.man.items()
                 if not k.startswith('__') and isinstance(v, dict)]
        done = [k for k, v in items if v.get('status') == 'done']
        bad = [k for k, v in items if v.get('status') != 'done']
        return done, bad


def run_signature():
    """Identity of this configuration. Checkpoints are only reused across runs when it matches.

    Without this, a checkpoint set written under different settings - a different fold count,
    fingerprint size, or with the neural stages off - would be adopted silently and produce results
    that do not correspond to any single configuration.
    """
    payload = json.dumps({
        'train_bytes': os.path.getsize(os.path.join(DATA_DIR, 'train.csv')),
        'test_bytes': os.path.getsize(os.path.join(DATA_DIR, 'test.csv')),
        'folds': N_FOLDS, 'seed': SEED, 'targets': TARGET_TYPES,
        'fp': [MORGAN_BITS_R2, MORGAN_BITS_R3, AP_BITS, TT_BITS],
        'neural': bool(USE_NEURAL), 'nn_seeds': NN_SEED_COUNT,
    }, sort_keys=True)
    return hashlib.md5(payload.encode()).hexdigest()[:16]


SIGNATURE = run_signature()


def _import_previous_checkpoints():
    """Adopt a previous run's checkpoints when its output is attached as a dataset.

    A Kaggle commit starts with an empty /kaggle/working, so checkpoints from an earlier commit are
    only reachable if that run's output is attached as an input dataset. Attach the previous
    version's output and run again - nothing else to configure. Only checkpoint stores whose
    signature matches this configuration are adopted.
    """
    if os.path.exists(os.path.join(CKPT_DIR, 'manifest.json')):
        return 0
    cands = []
    for root in ['/kaggle/input']:
        if os.path.isdir(root):
            cands += glob.glob(os.path.join(root, '**', 'ckpt', 'manifest.json'), recursive=True)
    cands = [c for c in cands if os.path.abspath(os.path.dirname(c)) != os.path.abspath(CKPT_DIR)]
    if not cands:
        return 0
    import shutil
    for cand in sorted(cands, key=os.path.getmtime, reverse=True):
        src = os.path.dirname(cand)
        try:
            sig = json.load(open(cand)).get('__signature__')
        except Exception:
            continue
        if sig is None:
            # written before signatures existed; adopt it rather than discard hours of work
            log.warn(f'checkpoints at {src} predate signature tracking - adopting them. '
                     f'Delete them if they came from a different configuration.')
        elif sig != SIGNATURE:
            log.warn(f'ignoring checkpoints at {src}: signature {sig} != {SIGNATURE} '
                     f'(different configuration - reusing them would mix two setups)')
            continue
        n = 0
        for f in glob.glob(os.path.join(src, '*.pkl')) + [cand]:
            if os.path.exists(f):
                shutil.copy2(f, os.path.join(CKPT_DIR, os.path.basename(f))); n += 1
        fsrc = os.path.join(src, 'folds')
        if os.path.isdir(fsrc):
            for f in glob.glob(os.path.join(fsrc, '*.pkl')):
                shutil.copy2(f, os.path.join(FOLD_DIR, os.path.basename(f))); n += 1
        log.ok(f'imported {n} checkpoint file(s) from {src} (signature {sig})')
        return n
    return 0


_imported = _import_previous_checkpoints()
CK = Ckpt(CKPT_DIR)
_local_sig = CK.man.get('__signature__')
if _local_sig and _local_sig != SIGNATURE:
    log.warn(f'existing checkpoints were written under signature {_local_sig}, this run is '
             f'{SIGNATURE} - discarding them rather than mixing two configurations')
    CK.man = {}
CK.man['__signature__'] = SIGNATURE
CK._flush()


STAGE_TITLES = {
    '01_invariance_selftest': 'POLYMER INVARIANCE SELF-TEST',
    '02_load_normalise': 'LOADING AND NORMALISING DATA',
    '03_features': 'FEATURE ENGINEERING',
    '04_pretrain_corpus': 'PI1M PRETRAINING CORPUS',
    '05_pretrain': 'PI1M ENCODER PRETRAINING',
    '06_embeddings': 'ENCODER EMBEDDINGS',
    '07_partner_physics': 'TRUE PARTNER FEATURES + PHYSICS',
    '09_folds': 'GROUPED CROSS-VALIDATION FOLDS',
    '10_gbdt': 'GRADIENT BOOSTING',
    '10b_reference_models': 'REFERENCE MODELS FOR EXPLANATION',
    '11_multitask_nn': 'MULTI-TASK NEURAL NETWORK',
    '12_finetune': 'FINE-TUNING THE PI1M ENCODER',
    '13_cnn': 'SMILES 1D-CNN',
    '14_stack': 'RIDGE STACKING',
    '15_physics': 'PHYSICS BLENDING',
    '16_invariance_audit': 'INVARIANCE AUDIT (END TO END)',
    '17_explain': 'EXPLAINABILITY',
}


# Diagnostics. Valuable, but they must never stop the notebook from producing a submission -
# the predictions are already final by the time these run.
NONFATAL_STAGES = {'16_invariance_audit', '17_explain'}


def run_stage(name, fn, key=''):
    """Execute fn() unless an artifact with a matching input key already exists."""
    title = STAGE_TITLES.get(name, name.upper())
    log.set_stage(name)
    if name in SKIP_STAGES and not CK.has(name, key):
        log.warn(f'STAGE SKIPPED      {name:<22} (PPP_SKIP) - no checkpoint written, '
                 f'a later run can still produce it')
        return None
    if CK.has(name, key):
        e = CK.man[name]
        secs = e.get('meta', {}).get('seconds')
        saved = f', {secs/60:.1f} min saved' if isinstance(secs, (int, float)) else ''
        log.ok(f'CHECKPOINT LOADED  {name:<22} <- {os.path.basename(CK.resolve(name))}{saved}')
        return CK.load(name)
    log.header(title)
    t0 = time.time()
    try:
        out = fn()
    except Exception as e:
        CK.fail(name, traceback.format_exc())
        log.error(f'STAGE FAILED  {name}: {type(e).__name__}: {e}')
        if name in NONFATAL_STAGES:
            log.warn(f'{name} is a diagnostic - continuing so the submission is still written')
            log.warn(traceback.format_exc().strip().split(chr(10))[-1])
            return None
        log.error('re-run the notebook - completed stages are skipped and this one is retried')
        raise
    dt = time.time() - t0
    CK.put(name, out, key, {'seconds': round(dt, 1)})
    log.ok(f'CHECKPOINT SAVED   {name:<22} -> {os.path.basename(CK.path(name))} '
           f'({dt/60:.1f} min)')
    return out


# ---- fine-grained per-fold / per-epoch checkpoints ----
def fpath(tag):
    return os.path.join(FOLD_DIR, tag.replace('/', '__') + '.pkl')


def fsave(tag, obj):
    _atomic_write(fpath(tag), lambda p: pickle.dump(obj, open(p, 'wb'), protocol=4))


def fload(tag):
    with open(fpath(tag), 'rb') as h:
        return pickle.load(h)


def fhas(tag):
    return os.path.exists(fpath(tag))


def cpu_state(sd):
    return {k: v.detach().cpu().clone() for k, v in sd.items()}

In [ ]:
log.set_stage('startup')
log.header('ENVIRONMENT AND RESUME SCAN')
log.info(f'data dir   : {DATA_DIR}')
log.info(f'work dir   : {WORK_DIR}')
log.info(f'ckpt dir   : {CKPT_DIR}')
log.info(f'PI1M       : {PI1M_PATH or "NOT FOUND"}')
log.info(f'aux smiles : {AUX_PATH or "NOT FOUND"}')
log.info(f'device     : {DEVICE}'
         + (f'  ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))
log.info(f'xgboost={HAVE_XGB}  catboost={HAVE_CB}  shap={HAVE_SHAP}')
log.info(f'neural stages (PI1M encoder, fine-tune, CNN): '
         f'{"ENABLED" if USE_NEURAL else "DISABLED - CPU backbone only"}')
if SKIP_STAGES:
    log.warn(f'skipping this run: {sorted(SKIP_STAGES)}')
log.info(f'budget     : {BUDGET["total"]/3600:.1f} h total')

_done, _bad = CK.summary()
log.info('')
log.info(f'{"stage":<24}{"status":<12}{"artifact"}')
log.info('-' * 60)
for _k in STAGE_TITLES:
    if CK.man.get(_k, {}).get('status') == 'done' and CK.resolve(_k):
        _s = CK.man[_k].get('meta', {}).get('seconds')
        _t = f'{_s/60:.1f} min' if isinstance(_s, (int, float)) else ''
        log.info(f'{_k:<24}{"RESTORED":<12}{os.path.basename(CK.resolve(_k))}  {_t}')
    elif _k in CK.man:
        log.info(f'{_k:<24}{"RETRY":<12}(previously failed)')
    elif _k in SKIP_STAGES:
        log.info(f'{_k:<24}{"SKIPPED":<12}(PPP_SKIP)')
    else:
        log.info(f'{_k:<24}{"PENDING":<12}')
log.info('-' * 60)
log.info(f'{len(_done)} stage(s) restored, {len(STAGE_TITLES) - len(_done)} to run')

_nf = len(glob.glob(os.path.join(FOLD_DIR, '*.pkl')))
if _nf:
    _by = {}
    for _f in glob.glob(os.path.join(FOLD_DIR, '*.pkl')):
        _by[os.path.basename(_f).split('__')[0]] = _by.get(os.path.basename(_f).split('__')[0], 0) + 1
    log.info(f'fold/epoch checkpoints: {_nf} file(s)  '
             + '  '.join(f'{k}={v}' for k, v in sorted(_by.items())))
else:
    log.info('fold/epoch checkpoints: none')
log.info('')

## 1. Polymer invariance

A polymer SMILES carries two invariances that a naive model does not respect.

**Translation.** `*CCO*` and `*OCC*` and `*COC*` describe the same chain; only the point at which
the repeating loop was cut differs. Closing the two `*` ends into a macrocycle removes the cut
point entirely, and RDKit's canonical SMILES of that macrocycle is then a translation-invariant
key — `ring_key`.

**Repetition.** Writing the repeat unit once, twice or three times describes the same polymer,
but doubles every extensive descriptor and doubles the string length. `reduce_repeat` finds the
smallest repeat unit whose k-fold concatenation reproduces the molecule exactly, by cutting the
backbone at every period-aligned position and verifying the reconstruction. Verification is what
makes it safe: a candidate is accepted only if rebuilding from it returns the identical canonical
SMILES, so a merely-similar fragment is rejected.

Composing them gives `poly_key = ring_key(reduce_repeat(smiles))`, invariant to both at once.
The self-test below builds dimers and trimers of real competition molecules, re-randomises their
atom ordering, and **measures** how often the key survives.

It reports a rate rather than asserting perfection. RDKit's canonical ranking is not always
independent of input atom order for a molecule assembled by graph surgery, and how often that
bites depends on the RDKit build, so a small number of misses is expected on some versions. Each
miss costs a little partner coverage for that one molecule and nothing else, which is not worth
killing a long run over. A rate below `MIN_INVARIANCE` does stop the run, because that would mean
the construction is wrong rather than imprecise.

Measured on this dataset: 10,605 raw strings collapse to 8,990 canonical molecules and 8,978
`poly_key` classes; 47 competition molecules are already written as oligomers of a smaller unit.

In [ ]:
def _stars(m):
    return [a.GetIdx() for a in m.GetAtoms() if a.GetAtomicNum() == 0]


def _nbr(m, idx):
    nb = list(m.GetAtomWithIdx(idx).GetNeighbors())
    return nb[0].GetIdx() if nb else None


def _strip_iso(m):
    for a in m.GetAtoms():
        if a.GetAtomicNum() == 0:
            a.SetIsotope(0)
    return m


def _stable_smiles(mol, fallback=''):
    """Canonical SMILES via a parse round-trip.

    RDKit's canonical ranking is not always independent of the input atom order once a molecule
    has been built by graph surgery (AddBond/RemoveAtom) rather than parsed - aromaticity and ring
    perception can settle differently. Writing the molecule out and reading it back makes the
    perception depend on the graph alone, which is what the invariance keys need. The effect is
    version-dependent, so this matters more on some RDKit builds than others.
    """
    try:
        smi = Chem.MolToSmiles(mol)
    except Exception:
        return fallback
    m2 = Chem.MolFromSmiles(smi)
    if m2 is None:
        return smi
    try:
        return Chem.MolToSmiles(m2)
    except Exception:
        return smi


def canon(smi):
    m = Chem.MolFromSmiles(smi)
    return _stable_smiles(m, smi) if m is not None else smi


def make_oligomer(smi, k):
    """Concatenate the repeat unit k times head-to-tail. Returns smi unchanged on any failure."""
    if k <= 1:
        return smi
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return smi
    st = _stars(m)
    if len(st) != 2:
        return smi
    n = m.GetNumAtoms()
    combo = m
    for _ in range(k - 1):
        combo = Chem.CombineMols(combo, m)
    rw = Chem.RWMol(combo)
    drop = []
    for i in range(k - 1):
        tail, head = st[1] + i * n, st[0] + (i + 1) * n
        t_n, h_n = _nbr(rw, tail), _nbr(rw, head)
        if t_n is None or h_n is None:
            return smi
        bt = rw.GetBondBetweenAtoms(tail, t_n).GetBondType()
        rw.AddBond(t_n, h_n, bt)
        drop += [tail, head]
    for idx in sorted(drop, reverse=True):
        rw.RemoveAtom(idx)
    out = rw.GetMol()
    try:
        Chem.SanitizeMol(out)
        return _stable_smiles(out, smi)
    except Exception:
        return smi


def ring_key(smi):
    """Translation-invariant key: close the repeat unit into a macrocycle, then canonicalise."""
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return smi
    st = _stars(m)
    if len(st) != 2:
        return _stable_smiles(m, smi)
    rw = Chem.RWMol(m)
    a, b = _nbr(rw, st[0]), _nbr(rw, st[1])
    if a is None or b is None or a == b or rw.GetBondBetweenAtoms(a, b) is not None:
        return _stable_smiles(m, smi)
    bt = rw.GetBondBetweenAtoms(st[0], a).GetBondType()
    rw.AddBond(a, b, bt)
    for idx in sorted(st, reverse=True):
        rw.RemoveAtom(idx)
    out = rw.GetMol()
    try:
        Chem.SanitizeMol(out)
        return _stable_smiles(out, smi)
    except Exception:
        return _stable_smiles(m, smi)


def reduce_repeat(smi, max_k=6):
    """Smallest repeat unit whose k-mer reproduces this molecule exactly."""
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return smi
    st = _stars(m)
    if len(st) != 2:
        return smi
    try:
        path = list(Chem.rdmolops.GetShortestPath(m, st[0], st[1]))
    except Exception:
        return smi
    bb = path[1:-1]
    L = len(bb)
    if L < 2:
        return smi
    target = _stable_smiles(m, smi)
    for k in range(max_k, 1, -1):
        if L % k:
            continue
        p = L // k
        for j in range(1, k):                       # every period-aligned cut
            bond = m.GetBondBetweenAtoms(bb[j * p - 1], bb[j * p])
            if bond is None:
                continue
            try:
                frags = Chem.FragmentOnBonds(m, [bond.GetIdx()], addDummies=True)
                pieces = Chem.GetMolFrags(frags, asMols=True, sanitizeFrags=True)
            except Exception:
                continue
            if len(pieces) != 2:
                continue
            for pc in pieces:
                cand = _stable_smiles(_strip_iso(Chem.RWMol(pc)))
                if cand.count('*') != 2:
                    continue
                if make_oligomer(cand, k) == target:   # verified, not guessed
                    return reduce_repeat(cand, max_k)
    return smi


def poly_key(smi):
    """Invariant to SMILES translation AND to monomer/dimer/trimer repetition."""
    return ring_key(reduce_repeat(smi))


def norm_smiles(smi):
    """Canonical, smallest-repeat-unit form of a polymer SMILES. Featurisation input."""
    return canon(reduce_repeat(smi))


def rand_smiles_many(smi, n, seed):
    """n alternative spellings of the same molecule.

    Uses RDKit's *seeded* random-SMILES API. `MolToSmiles(doRandom=True)` draws from RDKit's own
    global RNG, which no Python-level seed reaches, and would make a resumed run diverge from an
    uninterrupted one.
    """
    if n <= 0:
        return []
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return [smi] * n
    try:
        out = list(Chem.MolToRandomSmilesVect(m, n, randomSeed=int(seed) % (2 ** 31 - 1)))
    except Exception:
        out = []
    return (out + [smi] * n)[:n]


def rand_smiles(smi, seed=SEED):
    return rand_smiles_many(smi, 1, seed)[0]


def repr_variants(smi, n_rand=3, oligo=(2,), seed=SEED):
    """Valid alternative spellings of the same polymer, for augmentation and TTA."""
    out = [norm_smiles(smi)]
    seen = set(out)
    for k in oligo:
        o = make_oligomer(out[0], k)
        if o not in seen:
            seen.add(o); out.append(o)
    for r in rand_smiles_many(out[0], n_rand * 3, seed):
        if len(out) >= 1 + len(oligo) + n_rand:
            break
        if r not in seen:
            seen.add(r); out.append(r)
    return out

In [ ]:
MIN_INVARIANCE = 0.90      # below this, something is genuinely broken rather than an edge case


def _invariance_selftest():
    """Measure invariance and report it. Only a catastrophic rate aborts the run.

    A handful of misses is an RDKit canonicalisation edge case and varies by RDKit build; it costs
    a little partner coverage on those molecules and nothing else. Killing a 12-hour run over it
    would be the wrong trade, so the failures are logged by name, recorded in report.json, and the
    run continues. A rate below MIN_INVARIANCE means the construction itself is wrong, and that
    does stop the run.
    """
    log.sub('invariance self-test')
    probe = pd.read_csv(f'{DATA_DIR}/train.csv').smiles.drop_duplicates()
    rng = np.random.default_rng(SEED)
    sample = list(rng.choice(probe.values, min(200, len(probe)), replace=False))
    n = ok_oligo = ok_both = ok_rot = n_rot = 0
    misses = {'translation': [], 'repetition': [], 'both': []}
    for s in sample:
        base = poly_key(s)
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        for k in (2, 3):
            o = make_oligomer(s, k)
            if o == s:
                continue
            n += 1
            if poly_key(o) == base:
                ok_oligo += 1
            elif len(misses['repetition']) < 10:
                misses['repetition'].append(s)
            if poly_key(rand_smiles(o, SEED + k)) == base:
                ok_both += 1
            elif len(misses['both']) < 10:
                misses['both'].append(s)
        for r in rand_smiles_many(s, 2, SEED + 99):
            n_rot += 1
            if poly_key(r) == base:
                ok_rot += 1
            elif len(misses['translation']) < 10:
                misses['translation'].append(s)

    rates = dict(translation=ok_rot / max(n_rot, 1),
                 repetition=ok_oligo / max(n, 1),
                 both=ok_both / max(n, 1))
    log.metric(f'  translation (random SMILES) : {ok_rot}/{n_rot}   ({rates["translation"]*100:.2f}%)')
    log.metric(f'  repetition  (dimer, trimer) : {ok_oligo}/{n}   ({rates["repetition"]*100:.2f}%)')
    log.metric(f'  both at once                : {ok_both}/{n}   ({rates["both"]*100:.2f}%)')

    worst = min(rates.values())
    for kind, bad in misses.items():
        if bad:
            log.warn(f'  {len(bad)} {kind} miss(es), e.g. {bad[0]}')
    if worst >= 1.0:
        log.ok('poly_key is exactly invariant to both Round-3 invariances')
    else:
        log.warn(f'poly_key is invariant on {worst*100:.2f}% of probes, not 100% - the misses are '
                 f'RDKit canonicalisation edge cases and cost only partner coverage on those '
                 f'molecules. Continuing.')
    assert worst >= MIN_INVARIANCE, (
        f'invariance rate {worst:.3f} is below {MIN_INVARIANCE} - the key construction is broken, '
        f'not merely imprecise')
    return dict(rates=rates, rotation_cases=n_rot, rotation_ok=ok_rot,
                oligomer_cases=n, oligomer_ok=ok_oligo, oligomer_plus_rotation_ok=ok_both,
                example_misses=misses)


inv_selftest = run_stage('01_invariance_selftest', _invariance_selftest)

## 2. Load data and normalise every molecule

Each SMILES is rewritten to its canonical smallest-repeat-unit form before anything else touches
it, and `poly_key` is attached for joining and fold grouping. Duplicate `(poly_key, target_type)`
rows are averaged, so a molecule written two different ways cannot appear twice with conflicting
weight.

In [ ]:
def _load():
    tr = pd.read_csv(f'{DATA_DIR}/train.csv')
    te = pd.read_csv(f'{DATA_DIR}/test.csv')
    log.info(f'raw train {tr.shape}  test {te.shape}')

    tr = tr.drop_duplicates(subset=['smiles', 'target_type', 'target']).reset_index(drop=True)
    te = te.reset_index(drop=True)

    uniq = sorted(set(tr.smiles) | set(te.smiles))
    log.info(f'normalising {len(uniq)} unique SMILES (canonical + smallest repeat unit)...')
    t0 = time.time()
    nmap, kmap = {}, {}
    for i, s in enumerate(uniq):
        nmap[s] = norm_smiles(s)
        kmap[s] = ring_key(nmap[s])
        if (i + 1) % 2500 == 0:
            log.info(f'  {i+1}/{len(uniq)}  ({time.time()-t0:.0f}s)')
    n_red = sum(1 for s in uniq if canon(nmap[s]) != canon(s))
    log.metric(f'  raw {len(uniq)} -> canonical {len({canon(s) for s in uniq})} '
               f'-> poly_key {len(set(kmap.values()))}')
    log.metric(f'  {n_red} molecule(s) were written as oligomers and were reduced')

    for d in (tr, te):
        d['nsmiles'] = d.smiles.map(nmap)
        d['pkey'] = d.smiles.map(kmap)

    # average duplicate (pkey, target_type) observations
    before = len(tr)
    tr = (tr.groupby(['pkey', 'target_type'], as_index=False)
            .agg(smiles=('smiles', 'first'), nsmiles=('nsmiles', 'first'),
                 target=('target', 'mean'), n_obs=('target', 'size'))
            .reset_index(drop=True))
    log.info(f'  train rows {before} -> {len(tr)} after averaging duplicate (pkey,type)')

    for tt in TARGET_TYPES:
        a = (tr.target_type == tt).sum(); b = (te.target_type == tt).sum()
        log.info(f'  {tt:<4} train {a:>5}  test {b:>5}')
    return tr, te, nmap, kmap


train_df, test_df, NMAP, KMAP = run_stage('02_load_normalise', _load)
y_all = train_df.target.values.astype(np.float64)
t_all = train_df.target_type.map(TASK_MAP).values.astype(np.int64)
t_test = test_df.target_type.map(TASK_MAP).values.astype(np.int64)

## 3. Featurisation

Three groups, all computed on the normalised (canonical, smallest-repeat-unit) SMILES:

- **RDKit descriptors and fingerprints** — the full `CalcMolDescriptors` set, Morgan r2/r3,
  atom-pair, topological-torsion, MACCS, and SMARTS group counts.
- **Polymer topology** — the backbone is the shortest path between the two `*` markers; everything
  hanging off it is side chain. Backbone length, backbone aromatic and rotatable fractions,
  side-chain heavy-atom count and molecular weight, longest side chain, and branch-point count.
  These separate a stiff aromatic backbone with short side groups (high Tg) from a flexible
  backbone with long alkyl tails (low Tg), which no whole-molecule descriptor expresses.
- **Intensive normalisations** — extensive descriptors divided by heavy-atom count. A descriptor
  like molecular weight doubles when a repeat unit is written twice; its per-atom version does
  not. Reduction already removes that failure mode, but carrying both forms means the models do
  not have to rely on reduction having fired.

In [ ]:
GROUP_SMARTS = {
    'aromatic_6': '[a]1[a][a][a][a][a]1', 'aromatic_5': '[a]1[a][a][a][a]1',
    'amide': '[NX3][CX3](=[OX1])', 'ester': '[CX3](=[OX1])[OX2]',
    'ether': '[OD2]([#6])[#6]', 'hydroxyl': '[OX2H]', 'carbonyl': '[CX3]=[OX1]',
    'carboxyl': '[CX3](=[OX1])[OX2H1]', 'sulfonyl': '[#16X4](=[OX1])(=[OX1])',
    'imide': '[CX3](=[OX1])[NX3][CX3](=[OX1])', 'urea': '[NX3][CX3](=[OX1])[NX3]',
    'cyano': '[CX2]#[NX1]', 'nitro': '[NX3+](=O)[O-]', 'fluorine': '[F]',
    'chlorine': '[Cl]', 'bromine': '[Br]', 'silicon': '[Si]', 'phosphorus': '[P]',
    'double_bond': '[CX3]=[CX3]', 'triple_bond': '[CX2]#[CX2]', 'epoxide': 'C1OC1',
    'azo': '[NX2]=[NX2]', 'thioether': '[#16X2]([#6])[#6]', 'amine_primary': '[NX3H2]',
    'amine_secondary': '[NX3H1]([#6])[#6]', 'amine_tertiary': '[NX3]([#6])([#6])[#6]',
    'phenol': '[OX2H][c]', 'vinyl': '[CX3]=[CX3H1]', 'methyl': '[CH3]',
    'trifluoromethyl': '[CX4](F)(F)F', 'anhydride': '[CX3](=[OX1])[OX2][CX3](=[OX1])',
    'sulfone_aryl': '[c][#16X4](=[OX1])(=[OX1])[c]', 'carbonate': '[OX2][CX3](=[OX1])[OX2]',
    'imine': '[CX3]=[NX2]', 'siloxane': '[Si][OX2][Si]',
}
GROUP_PATTERNS = {k: p for k, s in GROUP_SMARTS.items() if (p := Chem.MolFromSmarts(s)) is not None}

# descriptors that scale with molecule size; also stored per heavy atom
EXTENSIVE = ['MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'LabuteASA',
             'TPSA', 'MolMR', 'BertzCT', 'HeavyAtomCount', 'NHOHCount', 'NOCount',
             'NumHAcceptors', 'NumHDonors', 'NumRotatableBonds', 'RingCount',
             'NumAromaticRings', 'NumAliphaticRings', 'NumSaturatedRings', 'Chi0', 'Chi1',
             'Kappa1', 'Kappa2', 'Kappa3', 'HallKierAlpha']


def topology_features(mol):
    """Backbone / side-chain decomposition of a two-star polymer repeat unit."""
    f = {}
    if mol is None:
        return f
    n_heavy = max(mol.GetNumHeavyAtoms(), 1)
    st = _stars(mol)
    f['n_star'] = len(st)
    if len(st) != 2:
        return f
    try:
        path = list(Chem.rdmolops.GetShortestPath(mol, st[0], st[1]))
    except Exception:
        return f
    bb = [i for i in path[1:-1]]
    bset = set(bb)
    L = len(bb)
    if L == 0:
        return f
    f['bb_len'] = L
    f['bb_frac'] = L / n_heavy
    f['side_frac'] = 1.0 - f['bb_frac']
    f['bb_arom'] = sum(1 for i in bb if mol.GetAtomWithIdx(i).GetIsAromatic()) / L
    f['bb_ring'] = sum(1 for i in bb if mol.GetAtomWithIdx(i).IsInRing()) / L
    f['bb_hetero'] = sum(1 for i in bb if mol.GetAtomWithIdx(i).GetAtomicNum() not in (6, 0)) / L
    rot = 0
    for a, b in zip(bb[:-1], bb[1:]):
        bd = mol.GetBondBetweenAtoms(a, b)
        if bd is not None and bd.GetBondType() == Chem.BondType.SINGLE and not bd.IsInRing():
            rot += 1
    f['bb_rot_frac'] = rot / max(L - 1, 1)

    # branch points: backbone atoms carrying a substituent that leaves the backbone
    branch = 0
    side_roots = []
    for i in bb:
        for nb in mol.GetAtomWithIdx(i).GetNeighbors():
            j = nb.GetIdx()
            if j not in bset and nb.GetAtomicNum() != 0:
                branch += 1
                side_roots.append((i, j))
    f['n_branch'] = branch
    f['branch_per_bb'] = branch / L

    # side chains: everything reachable without re-entering the backbone
    sizes, weights = [], []
    for i, j in side_roots:
        seen, stack = {j}, [j]
        while stack:
            cur = stack.pop()
            for nb in mol.GetAtomWithIdx(cur).GetNeighbors():
                k = nb.GetIdx()
                if k not in seen and k not in bset and nb.GetAtomicNum() != 0:
                    seen.add(k); stack.append(k)
        sizes.append(len(seen))
        weights.append(sum(mol.GetAtomWithIdx(k).GetMass() for k in seen))
    f['n_side'] = len(sizes)
    f['side_atoms'] = float(sum(sizes))
    f['side_mw'] = float(sum(weights))
    f['side_max'] = float(max(sizes)) if sizes else 0.0
    f['side_mean'] = float(np.mean(sizes)) if sizes else 0.0
    f['side_mw_per_bb'] = f['side_mw'] / L
    f['side_atoms_per_bb'] = f['side_atoms'] / L
    return f


def custom_features(mol, smi):
    f = {}
    if mol is None:
        return f
    try:
        n = max(mol.GetNumAtoms(), 1)
        f['smi_len'] = len(smi)
        f['n_atoms'] = mol.GetNumAtoms(); f['n_heavy'] = mol.GetNumHeavyAtoms()
        f['n_bonds'] = mol.GetNumBonds()
        f['n_rings'] = mol.GetRingInfo().NumRings()
        f['n_arom_rings'] = rdMolDescriptors.CalcNumAromaticRings(mol)
        f['n_aliph_rings'] = rdMolDescriptors.CalcNumAliphaticRings(mol)
        f['arom_ratio'] = f['n_arom_rings'] / max(f['n_rings'], 1)
        f['ring_ratio'] = f['n_rings'] / n
        f['n_rot'] = Lipinski.NumRotatableBonds(mol)
        f['rot_ratio'] = f['n_rot'] / max(f['n_bonds'], 1)
        f['n_het'] = Lipinski.NumHeteroatoms(mol); f['het_ratio'] = f['n_het'] / n
        f['n_hbd'] = Lipinski.NumHDonors(mol); f['n_hba'] = Lipinski.NumHAcceptors(mol)
        f['hbd_ratio'] = f['n_hbd'] / n; f['hba_ratio'] = f['n_hba'] / n
        f['fsp3'] = rdMolDescriptors.CalcFractionCSP3(mol)
        cj = sum(1 for b in mol.GetBonds() if b.GetIsConjugated())
        f['n_conj_bonds'] = cj; f['conj_ratio'] = cj / max(f['n_bonds'], 1)
        nums = [a.GetAtomicNum() for a in mol.GetAtoms()]
        for z, nm in [(6, 'C'), (7, 'N'), (8, 'O'), (9, 'F'), (16, 'S'),
                      (17, 'Cl'), (35, 'Br'), (14, 'Si'), (15, 'P')]:
            f[f'n_{nm}'] = nums.count(z); f[f'fr_{nm}'] = nums.count(z) / n
        try:
            AllChem.ComputeGasteigerCharges(mol)
            ch = [a.GetDoubleProp('_GasteigerCharge') for a in mol.GetAtoms()]
            ch = [c for c in ch if np.isfinite(c)]
            if ch:
                f['ch_mean'] = float(np.mean(ch)); f['ch_std'] = float(np.std(ch))
                f['ch_min'] = float(np.min(ch)); f['ch_max'] = float(np.max(ch))
                f['ch_range'] = f['ch_max'] - f['ch_min']
                f['ch_absum'] = float(np.sum(np.abs(ch)))
                f['ch_absum_per_atom'] = f['ch_absum'] / n
        except Exception:
            pass
        for nm, fn in [('balaban_j', Descriptors.BalabanJ), ('bertz_ct', Descriptors.BertzCT),
                       ('ipc', Descriptors.Ipc)]:
            try:
                v = fn(mol)
                f[nm] = float(v) if np.isfinite(v) else 0.0
            except Exception:
                pass
    except Exception:
        pass
    return {k: (0.0 if v is None or (isinstance(v, float) and not np.isfinite(v)) else v)
            for k, v in f.items()}


def featurize_batch(smiles_list):
    n = len(smiles_list); t0 = time.time(); every = max(1, n // 12)
    rd_l, m2_l, m3_l, ap_l, tt_l, mc_l, cu_l, gr_l, tp_l = ([] for _ in range(9))
    apg = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=AP_BITS)
    ttg = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=TT_BITS)
    log.info(f'featurising {n} molecules...')
    for i, smi in enumerate(smiles_list):
        if (i + 1) % every == 0:
            el = time.time() - t0
            log.info(f'  {i+1}/{n} ({100*(i+1)/n:.0f}%)  eta {(n-i-1)/max((i+1)/el, .01):.0f}s')
        mol = Chem.MolFromSmiles(smi)
        try:
            d = Descriptors.CalcMolDescriptors(mol) if mol is not None else {}
            row = {k: (0.0 if v is None or (isinstance(v, float) and not np.isfinite(v)) else float(v))
                   for k, v in d.items()}
        except Exception:
            row = {}
        nh = max(mol.GetNumHeavyAtoms(), 1) if mol is not None else 1
        for k in EXTENSIVE:                       # intensive twin of each extensive descriptor
            if k in row:
                row[f'{k}_per_atom'] = row[k] / nh
        rd_l.append(row)
        if mol is not None:
            m2_l.append(np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=MORGAN_BITS_R2), dtype=np.uint8))
            m3_l.append(np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=MORGAN_BITS_R3), dtype=np.uint8))
            ap_l.append((apg.GetFingerprintAsNumPy(mol) > 0).astype(np.uint8))
            tt_l.append((ttg.GetFingerprintAsNumPy(mol) > 0).astype(np.uint8))
            mc_l.append(np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.uint8))
        else:
            m2_l.append(np.zeros(MORGAN_BITS_R2, np.uint8)); m3_l.append(np.zeros(MORGAN_BITS_R3, np.uint8))
            ap_l.append(np.zeros(AP_BITS, np.uint8)); tt_l.append(np.zeros(TT_BITS, np.uint8))
            mc_l.append(np.zeros(167, np.uint8))
        cu_l.append(custom_features(mol, smi))
        tp_l.append(topology_features(mol))
        gr_l.append({f'grp_{k}': len(mol.GetSubstructMatches(p)) if mol is not None else 0
                     for k, p in GROUP_PATTERNS.items()})
    log.info(f'  featurisation done in {time.time()-t0:.0f}s')
    df_rd = pd.DataFrame(rd_l).add_prefix('rd_')
    df_cu = pd.DataFrame(cu_l).add_prefix('po_')
    df_tp = pd.DataFrame(tp_l).add_prefix('tp_')
    parts = [df_rd,
             pd.DataFrame(np.array(m2_l), columns=[f'mfp2_{i}' for i in range(MORGAN_BITS_R2)]),
             pd.DataFrame(np.array(m3_l), columns=[f'mfp3_{i}' for i in range(MORGAN_BITS_R3)]),
             pd.DataFrame(np.array(ap_l), columns=[f'ap_{i}' for i in range(AP_BITS)]),
             pd.DataFrame(np.array(tt_l), columns=[f'tt_{i}' for i in range(TT_BITS)]),
             pd.DataFrame(np.array(mc_l), columns=[f'mac_{i}' for i in range(167)]),
             df_cu, df_tp, pd.DataFrame(gr_l)]
    return pd.concat(parts, axis=1)


def clean_features(df):
    df = df.copy()
    fm = float(np.finfo(np.float32).max)
    num = df.select_dtypes(include=[np.number]).columns
    df[num] = df[num].clip(lower=-fm, upper=fm)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0.0, inplace=True)
    obj = df.select_dtypes(include=['object']).columns.tolist()
    if obj:
        df.drop(columns=obj, inplace=True)
    df.columns = [re.sub(r'[\[\]<,]', '_', c) for c in df.columns]
    return df

In [ ]:
def _build_features():
    uniq = pd.unique(pd.concat([train_df.nsmiles, test_df.nsmiles]))
    log.info(f'{len(uniq)} distinct normalised molecules to featurise')
    feats = clean_features(featurize_batch(list(uniq)))
    feats.index = uniq
    trf = feats.loc[train_df.nsmiles.values].reset_index(drop=True)
    tef = feats.loc[test_df.nsmiles.values].reset_index(drop=True)
    const = trf.columns[trf.nunique() <= 1].tolist()
    if const:
        trf.drop(columns=const, inplace=True)
        tef.drop(columns=[c for c in const if c in tef.columns], inplace=True)
        log.info(f'dropped {len(const)} constant columns')
    trf = trf.astype(np.float32); tef = tef.astype(np.float32)
    log.metric(f'feature matrix: train {trf.shape}  test {tef.shape}')
    del feats; gc.collect()
    return trf, tef


train_features, test_features = run_stage('03_features', _build_features)
BASE_COLS = list(train_features.columns)

## 4. PI1M pretraining — a polymer encoder trained from scratch

`PI1M.csv` carries 995,799 polymer SMILES, **100% of them with `*` end markers** and a length
distribution matching the labelled set (mean 46.8 vs 49.3 characters). It is the same chemical
domain as the targets. `smile_r3.csv` is 5.97M drug-like molecules with **0%** `*` markers — a
different domain — so it contributes only a small diversity sample.

A 6-layer encoder is trained from scratch with masked-language modelling. Nothing is downloaded;
the tokenizer is built from the corpus in this run and the weights come out of this run.

**Why this is the right lever.** Five properties have under 350 labels. A representation learned
from a million unlabelled polymers is what lets a 221-row target fit anything beyond memorised
fingerprint bits. TransPolymer established the recipe — pretrain on PI1M with SMILES-enumeration
augmentation — and reported measurable degradation when representations are not enumeration-
augmented.

Training is checkpointed **every 200 optimiser steps** with model, optimiser, scheduler, scaler
and RNG state, and it respects a wall-clock cap, so an interrupted run resumes mid-epoch rather
than restarting.

In [ ]:
SMI_REGEX = re.compile(
    r'(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\|/|:|~|@|\?|>|\*|\$|%[0-9]{2}|[0-9])')
PAD, MASK, UNK, BOS, EOS = 0, 1, 2, 3, 4
SPECIALS = ['<pad>', '<mask>', '<unk>', '<bos>', '<eos>']
MAX_LEN = 128

PRE_CFG = dict(d_model=256, n_head=8, n_layer=6, d_ff=1024, dropout=0.1,
               batch_size=256, lr=5e-4, warmup=1000, mask_prob=0.15,
               n_pi1m=300_000, n_aug=2, n_aux=100_000, comp_aug=5, log_every=200)


def smi_tokens(s):
    return SMI_REGEX.findall(s)


def _build_vocab(corpus):
    seen = {}
    for s in corpus:
        for t in smi_tokens(s):
            seen[t] = seen.get(t, 0) + 1
    toks = [t for t, c in sorted(seen.items(), key=lambda kv: (-kv[1], kv[0])) if c >= 5]
    vocab = {t: i for i, t in enumerate(SPECIALS)}
    for t in toks:
        vocab.setdefault(t, len(vocab))
    return vocab


def encode(s, vocab, max_len=MAX_LEN):
    ids = [BOS] + [vocab.get(t, UNK) for t in smi_tokens(s)][:max_len - 2] + [EOS]
    return ids + [PAD] * (max_len - len(ids))


def _aug_worker(args):
    """Return (canonical, *random variants) for one SMILES. Module-level for Pool pickling.

    The seed is derived from the SMILES itself, so the corpus is identical on every run and does
    not depend on how the work was split across processes.
    """
    smi, n = args
    out = [smi]
    if n <= 0:
        return out
    seed = int(hashlib.md5(smi.encode()).hexdigest()[:8], 16)
    seen = {smi}
    for r in rand_smiles_many(smi, n * 3, seed):
        if len(out) >= n + 1:
            break
        if r not in seen:
            seen.add(r); out.append(r)
    return out


def _parallel_aug(smiles, n_aug, tag):
    """Enumerate alternative spellings across processes, falling back to serial if Pool fails."""
    t0 = time.time()
    jobs = [(s, n_aug) for s in smiles]
    res = None
    try:
        import multiprocessing as mp
        nproc = max(1, min(4, (os.cpu_count() or 2)))
        if nproc > 1:
            # fork lets the workers inherit this module-level function without pickling it
            ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp
            with ctx.Pool(nproc) as pool:
                res = pool.map(_aug_worker, jobs, chunksize=2000)
    except Exception as e:
        log.warn(f'  parallel augmentation unavailable ({e}) - falling back to serial')
    if res is None:
        res = [_aug_worker(j) for j in jobs]
    flat = [s for group in res for s in group]
    log.info(f'  {tag}: {len(smiles)} -> {len(flat)} sequences in {time.time()-t0:.0f}s')
    return flat

In [ ]:
def _build_pretrain_corpus():
    if not USE_NEURAL:
        log.warn('neural stages disabled - skipping the PI1M corpus')
        return np.zeros((0, MAX_LEN), dtype=np.uint8), {t: i for i, t in enumerate(SPECIALS)}
    if PI1M_PATH is None:
        log.warn('PI1M.csv not found - pretraining corpus falls back to competition molecules only')
    parts = []

    comp = list(pd.unique(pd.concat([train_df.nsmiles, test_df.nsmiles])))
    parts.append(_parallel_aug(comp, PRE_CFG['comp_aug'], 'competition molecules'))

    if PI1M_PATH:
        pi = pd.read_csv(PI1M_PATH)
        col = pi.columns[0]
        pi = pi[col].dropna().astype(str)
        log.info(f'PI1M: {len(pi)} rows, sampling {PRE_CFG["n_pi1m"]}')
        pi = pi.sample(min(PRE_CFG['n_pi1m'], len(pi)), random_state=SEED).tolist()
        parts.append(_parallel_aug(pi, PRE_CFG['n_aug'], 'PI1M'))
        del pi; gc.collect()

    if AUX_PATH and PRE_CFG['n_aux'] > 0:
        try:
            ax = pd.read_csv(AUX_PATH, usecols=[0])
            col = ax.columns[0]
            ax = ax[col].dropna().astype(str)
            log.info(f'smile_r3: {len(ax)} rows, sampling {PRE_CFG["n_aux"]} for chemical diversity')
            parts.append(ax.sample(min(PRE_CFG['n_aux'], len(ax)), random_state=SEED).tolist())
            del ax; gc.collect()
        except Exception as e:
            log.warn(f'could not read auxiliary SMILES ({e}) - continuing without it')

    corpus = [s for p in parts for s in p]
    random.Random(SEED).shuffle(corpus)
    log.metric(f'pretraining corpus: {len(corpus)} sequences')

    vocab = _build_vocab(corpus[:200_000])
    for s in comp:                                  # every competition token must be in-vocab
        for t in smi_tokens(s):
            vocab.setdefault(t, len(vocab))
    log.metric(f'vocabulary: {len(vocab)} tokens')
    assert len(vocab) < 65535, 'vocabulary too large for uint16 storage'

    t0 = time.time()
    dtype = np.uint8 if len(vocab) < 256 else np.uint16
    arr = np.zeros((len(corpus), MAX_LEN), dtype=dtype)
    for i, s in enumerate(corpus):
        arr[i] = encode(s, vocab)
        if (i + 1) % 200_000 == 0:
            log.info(f'  tokenised {i+1}/{len(corpus)} ({time.time()-t0:.0f}s)')
    lens = (arr != PAD).sum(1)
    log.metric(f'token length: mean {lens.mean():.1f}  p95 {np.percentile(lens,95):.0f}  '
               f'truncated {100*(lens>=MAX_LEN).mean():.2f}%')
    log.metric(f'tokenised array {arr.shape} {arr.dtype} = {arr.nbytes/1e6:.0f} MB')
    return arr, vocab


pre_tokens, VOCAB = run_stage('04_pretrain_corpus', _build_pretrain_corpus,
                              key=json.dumps({k: PRE_CFG[k] for k in
                                              ['n_pi1m', 'n_aug', 'n_aux', 'comp_aug']}))
V = len(VOCAB)
INV_VOCAB = {i: t for t, i in VOCAB.items()}

In [ ]:
class SmilesEncoder(nn.Module):
    """Encoder-only transformer over SMILES tokens, with an MLM head for pretraining."""

    def __init__(self, vocab, d=256, heads=8, layers=6, ff=1024, p=0.1, max_len=MAX_LEN):
        super().__init__()
        self.d = d
        self.tok = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos = nn.Embedding(max_len, d)
        self.drop = nn.Dropout(p)
        self.norm = nn.LayerNorm(d)
        layer = nn.TransformerEncoderLayer(d, heads, ff, p, batch_first=True,
                                           norm_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, layers)
        self.mlm = nn.Linear(d, vocab)

    def hidden(self, x):
        pad = x.eq(PAD)
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.drop(self.norm(self.tok(x) + self.pos(pos)))
        return self.enc(h, src_key_padding_mask=pad), pad

    def pool(self, x):
        """Mean over real tokens - order-insensitive, so a re-spelled SMILES pools similarly."""
        h, pad = self.hidden(x)
        m = (~pad).unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1.0)

    def forward(self, x):
        h, _ = self.hidden(x)
        return self.mlm(h)


def mlm_batch(x, mask_prob, vocab_size, gen):
    """BERT masking: 80% <mask>, 10% random token, 10% unchanged. Labels -100 elsewhere."""
    labels = x.clone()
    special = (x == PAD) | (x == BOS) | (x == EOS)
    prob = torch.rand(x.shape, device=x.device, generator=gen)
    sel = (prob < mask_prob) & ~special
    labels[~sel] = -100
    r = torch.rand(x.shape, device=x.device, generator=gen)
    x = x.clone()
    x[sel & (r < 0.8)] = MASK
    rnd = torch.randint(len(SPECIALS), vocab_size, x.shape, device=x.device, generator=gen)
    hit = sel & (r >= 0.8) & (r < 0.9)
    x[hit] = rnd[hit]
    return x, labels


class _NullCtx:
    def __enter__(self): return None
    def __exit__(self, *a): return False


def _amp():
    """Autocast context and gradient scaler.

    PPP_FORCE_AMP=1 turns on CPU autocast when there is no GPU. Without it the whole
    mixed-precision path is dead code off-GPU, which is exactly how a dtype bug in it reaches
    Kaggle unseen. The smoke test sets it.
    """
    if not torch.cuda.is_available():
        if os.environ.get('PPP_FORCE_AMP') == '1':
            try:
                return torch.amp.autocast('cpu', dtype=torch.bfloat16), None
            except (AttributeError, TypeError):
                return _NullCtx(), None
        return _NullCtx(), None
    try:
        return torch.amp.autocast('cuda', dtype=torch.float16), torch.amp.GradScaler('cuda')
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(), torch.cuda.amp.GradScaler()

In [ ]:
def _pretrain():
    if not USE_NEURAL or len(pre_tokens) == 0:
        log.warn('neural stages disabled - no encoder will be pretrained')
        return dict(state=None, hist=[], steps=0, vocab_size=V, cfg={})
    cap = BUDGET['pretrain']
    tag = 'pretrain/state'
    model = SmilesEncoder(V, PRE_CFG['d_model'], PRE_CFG['n_head'], PRE_CFG['n_layer'],
                          PRE_CFG['d_ff'], PRE_CFG['dropout']).to(DEVICE)
    n_par = sum(p.numel() for p in model.parameters())
    log.info(f'encoder: {n_par/1e6:.1f}M parameters, vocab {V}, max_len {MAX_LEN}')

    opt = torch.optim.AdamW(model.parameters(), lr=PRE_CFG['lr'], weight_decay=0.01,
                            betas=(0.9, 0.98), eps=1e-6)
    bs = PRE_CFG['batch_size']
    steps_per_epoch = max(1, len(pre_tokens) // bs)
    total_steps = steps_per_epoch * 8                       # cosine horizon; wall clock ends it

    def lr_at(s):
        if s < PRE_CFG['warmup']:
            return s / max(PRE_CFG['warmup'], 1)
        q = (s - PRE_CFG['warmup']) / max(total_steps - PRE_CFG['warmup'], 1)
        return 0.5 * (1 + math.cos(math.pi * min(q, 1.0)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    autocast, scaler = _amp()
    gen = torch.Generator(device=DEVICE); gen.manual_seed(SEED)

    step, start_epoch, start_batch, hist = 0, 0, 0, []
    if fhas(tag):
        st = fload(tag)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
        sched.load_state_dict(st['sched'])
        if scaler is not None and st.get('scaler'):
            scaler.load_state_dict(st['scaler'])
        step, start_epoch, hist = st['step'], st['epoch'], st.get('hist', [])
        start_batch = st.get('batch', 0)
        log.ok(f'resumed pretraining at epoch {start_epoch}, batch {start_batch}, step {step}')

    idx_all = np.arange(len(pre_tokens))
    t_begin = time.time()
    stop = False
    for epoch in range(start_epoch, 8):
        if stop:
            break
        # every source of randomness is keyed on (epoch, batch), never on execution history,
        # so a resumed run reproduces an uninterrupted one exactly
        order = np.random.default_rng(SEED + epoch).permutation(idx_all)
        torch.manual_seed(SEED * 31 + epoch)
        first = start_batch if epoch == start_epoch else 0
        start_batch = 0
        model.train()
        run_loss, run_n, t_ep = 0.0, 0, time.time()
        for bi in range(first, steps_per_epoch):
            sl = order[bi * bs:(bi + 1) * bs]
            if len(sl) < 2:
                continue
            gen.manual_seed(SEED * 131 + epoch * 1_000_003 + bi)
            xb = torch.as_tensor(pre_tokens[sl].astype(np.int64), device=DEVICE)
            xin, lab = mlm_batch(xb, PRE_CFG['mask_prob'], V, gen)
            opt.zero_grad(set_to_none=True)
            with autocast:
                out = model(xin)
                loss = F.cross_entropy(out.reshape(-1, V), lab.reshape(-1), ignore_index=-100)
            if not torch.isfinite(loss):
                continue
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step(); step += 1
            run_loss += loss.item(); run_n += 1

            if step % PRE_CFG['log_every'] == 0:
                avg = run_loss / max(run_n, 1)
                ppl = math.exp(min(avg, 20))
                log.metric(f'  epoch {epoch+1} step {step} ({bi+1}/{steps_per_epoch}) '
                           f'loss {avg:.4f} ppl {ppl:.2f} lr {sched.get_last_lr()[0]:.2e}',
                           epoch=epoch + 1, step=step, loss=round(avg, 4))
                hist.append(dict(step=step, epoch=epoch + 1, loss=round(avg, 4)))
                run_loss, run_n = 0.0, 0
                fsave(tag, dict(model=cpu_state(model.state_dict()), opt=opt.state_dict(),
                                sched=sched.state_dict(),
                                scaler=scaler.state_dict() if scaler is not None else None,
                                step=step, epoch=epoch, batch=bi + 1, hist=hist))
                if time.time() - t_begin > cap or time_left() < BUDGET['finetune'] + 45 * 60:
                    log.warn(f'pretraining budget reached at step {step} - stopping cleanly')
                    stop = True
                    break
        log.metric(f'  epoch {epoch+1} finished in {(time.time()-t_ep)/60:.1f} min')
        fsave(tag, dict(model=cpu_state(model.state_dict()), opt=opt.state_dict(),
                        sched=sched.state_dict(),
                        scaler=scaler.state_dict() if scaler is not None else None,
                        step=step, epoch=epoch + 1, batch=0, hist=hist))

    final = cpu_state(model.state_dict())
    log.ok(f'pretraining done: {step} steps, {(time.time()-t_begin)/60:.1f} min, '
           f'final loss {hist[-1]["loss"] if hist else float("nan"):.4f}')
    return dict(state=final, hist=hist, steps=step, vocab_size=V,
                cfg={k: PRE_CFG[k] for k in ['d_model', 'n_head', 'n_layer', 'd_ff', 'dropout']})


pretrained = run_stage('05_pretrain', _pretrain)

In [ ]:
def _encoder_from(state):
    m = SmilesEncoder(V, PRE_CFG['d_model'], PRE_CFG['n_head'], PRE_CFG['n_layer'],
                      PRE_CFG['d_ff'], PRE_CFG['dropout']).to(DEVICE)
    m.load_state_dict(state)
    return m


def _embed(smiles_list, model, bs=512):
    model.eval()
    ids = np.stack([encode(s, VOCAB) for s in smiles_list]).astype(np.int64)
    out = np.zeros((len(ids), PRE_CFG['d_model']), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(ids), bs):
            xb = torch.as_tensor(ids[i:i + bs], device=DEVICE)
            out[i:i + bs] = model.pool(xb).float().cpu().numpy()
    return out


def _pretrain_embeddings():
    """Frozen encoder embeddings as extra columns for the tree models."""
    if pretrained.get('state') is None:
        log.warn('no pretrained encoder - the feature matrix gets no embedding columns')
        return pd.DataFrame(index=range(len(train_df))), pd.DataFrame(index=range(len(test_df)))
    model = _encoder_from(pretrained['state'])
    uniq = list(pd.unique(pd.concat([train_df.nsmiles, test_df.nsmiles])))
    emb = _embed(uniq, model)
    log.metric(f'embeddings {emb.shape}  (std {emb.std():.4f})')
    E = pd.DataFrame(emb, index=uniq, columns=[f'emb_{i}' for i in range(emb.shape[1])])
    tr = E.loc[train_df.nsmiles.values].reset_index(drop=True).astype(np.float32)
    te = E.loc[test_df.nsmiles.values].reset_index(drop=True).astype(np.float32)
    del model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return tr, te


emb_train, emb_test = run_stage('06_embeddings', _pretrain_embeddings)
if emb_train.shape[1]:
    train_features = pd.concat([train_features, emb_train], axis=1)
    test_features = pd.concat([test_features, emb_test], axis=1)
log.info(f'features with embeddings: train {train_features.shape}  test {test_features.shape}')

## 5. Co-observed partner features and physics

The six DFT properties are computed on the same molecules, so for a molecule whose target is
`eps`, its `nc` value is frequently **already present in train** — and it is a different property,
equally available at inference time, not a label for the row being predicted.

Keying that join on `poly_key` rather than the raw string is what makes it pay: mean partner
coverage on the test rows rises from **0.391 (raw) to 0.468 (poly_key)**, because two spellings
of the same polymer no longer look like two molecules.

**Physics.** Measured as direct, unfitted estimators on co-observed molecules:

```
ei  ≈ egc + eea    R² = 0.963      (fundamental gap)
eea ≈ ei  − egc    R² = 0.971
egb ≈ egc          R² = 0.892
eps ≈ nc²          R² = 0.843      (Maxwell relation)
```

An axis-aligned tree splits one feature at a time and cannot represent a sum of two columns, so
these must be handed over explicitly rather than discovered. They also route the 2,028 `egc`
labels into `ei` and `eea`, which have 222 and 221.

**Every engineered column records the labels it was built from in `USES`.** `true_egc` *is* the
answer for an `egc` row and `ph_ei = egc + eea` leaks for both `egc` and `eea` rows, so two
guards apply: `drop_leaky` removes offending columns for per-property models, and
`mask_rows_for_multitask` neutralises them per row for shared-trunk models, where a column that
leaks on `egc` rows is legitimate on `eps` rows.

In [ ]:
def physics_terms(raw):
    """The engineered columns, as a function of the raw partner values.

    Keeping this a pure function of `raw` means the identical definitions are reused by the
    invariance audit in section 14, which has to rebuild these columns for re-spelled molecules.
    Each entry is (column name, values, the property labels it was built from).
    """
    out = [(f'true_{q}', raw[q], [q]) for q in DFT_PROPS]
    out += [
        ('ph_ei',  raw['egc'] + raw['eea'],                 ['egc', 'eea']),   # R2 = 0.963
        ('ph_eea', raw['ei'] - raw['egc'],                  ['ei', 'egc']),    # R2 = 0.971
        ('ph_egb', raw['egc'],                              ['egc']),          # R2 = 0.892
        ('ph_eps', raw['nc'] ** 2,                          ['nc']),           # Maxwell
        ('ph_nc',  np.sqrt(np.clip(raw['eps'], 0, None)),   ['eps']),
        ('ph_gap', raw['egb'] - raw['egc'],                 ['egb', 'egc']),
        ('ph_mid', 0.5 * (raw['ei'] + raw['eea']),          ['ei', 'eea']),
    ]
    return out


def _partner_tables():
    """True partner values keyed on poly_key, plus the physics combinations of them.

    Returns the engineered columns rather than writing them into the global feature matrices, so a
    resumed run that skips this stage still ends up with exactly the same features.
    """
    truth = {q: train_df[train_df.target_type == q].groupby('pkey').target.mean()
             for q in DFT_PROPS}
    raw_tr = {q: train_df.pkey.map(truth[q]).values.astype(np.float64) for q in DFT_PROPS}
    raw_te = {q: test_df.pkey.map(truth[q]).values.astype(np.float64) for q in DFT_PROPS}
    for q in DFT_PROPS:
        log.info(f'  true_{q}: {len(truth[q])} molecules, '
                 f'{np.isfinite(raw_te[q]).mean()*100:.1f}% of test rows covered')

    uses, fills = {}, {}
    eng_tr, eng_te = {}, {}
    for (name, a, srcs), (_, b, _) in zip(physics_terms(raw_tr), physics_terms(raw_te)):
        fill = float(np.nanmean(np.where(np.isfinite(a), a, np.nan)))
        if not np.isfinite(fill):
            fill = 0.0
        fills[name] = fill; fills[f'{name}_ok'] = 0.0
        uses[name] = sorted(srcs); uses[f'{name}_ok'] = sorted(srcs)
        eng_tr[name] = np.where(np.isfinite(a), a, fill).astype(np.float32)
        eng_te[name] = np.where(np.isfinite(b), b, fill).astype(np.float32)
        eng_tr[f'{name}_ok'] = np.isfinite(a).astype(np.float32)
        eng_te[f'{name}_ok'] = np.isfinite(b).astype(np.float32)

    n_part = np.column_stack([np.isfinite(raw_te[q]) for q in DFT_PROPS]).sum(1)
    log.metric(f'test rows with >=1 true partner: {(n_part>0).mean()*100:.1f}%  '
               f'(mean {n_part.mean():.2f} partners)')
    return dict(uses=uses, fills=fills,
                truth={q: truth[q].to_dict() for q in DFT_PROPS},
                eng_tr=pd.DataFrame(eng_tr), eng_te=pd.DataFrame(eng_te))


PARTNER = run_stage('07_partner_physics', _partner_tables)
TRUTH = {q: pd.Series(PARTNER['truth'][q]) for q in DFT_PROPS}
USES = {k: set(v) for k, v in PARTNER['uses'].items()}
FILLS = PARTNER['fills']
ENG_COLS = list(PARTNER['eng_tr'].columns)
train_features = pd.concat([train_features, PARTNER['eng_tr']], axis=1)
test_features = pd.concat([test_features, PARTNER['eng_te']], axis=1)
FEAT_COLS = list(train_features.columns)
assert list(test_features.columns) == FEAT_COLS, 'train/test feature columns diverged'
log.metric(f'{len(ENG_COLS)} engineered columns -> {len(FEAT_COLS)} features total')


def drop_leaky(feat_df, target_type):
    """Per-property models: drop every column built from this target's own label."""
    bad = [c for c, s in USES.items() if target_type in s and c in feat_df.columns]
    return feat_df.drop(columns=bad)


def mask_rows_for_multitask(feat_df, target_types):
    """Shared-trunk models: neutralise per row, using the train fill so train and test match."""
    out = feat_df.copy()
    tt = np.asarray(target_types)
    for c, s in USES.items():
        if c in out.columns:
            m = np.isin(tt, list(s))
            if m.any():
                out.loc[m, c] = FILLS[c]
    return out

## 6. Leakage self-test

The notebook first proves the leak is **real** — `true_eps` reproduces the `eps` target exactly —
then proves each guard removes it. A guard that passes without the leak being demonstrable would
prove nothing. Execution stops if any check fails.

In [ ]:
log.set_stage('08_leakage_test')
log.header('LEAKAGE SELF-TEST')
_fail = []

for p in DFT_PROPS:
    m = (train_df.target_type == p).values
    if m.sum() and not np.allclose(train_features.loc[m, f'true_{p}'],
                                   train_df.loc[m, 'target'], atol=1e-5):
        _fail.append(f'{p}: true_{p} does not reproduce the target - the feature build is wrong')
log.info('(a) leak reproduced for every DFT property, as expected')

for p in TARGET_TYPES:
    kept = drop_leaky(train_features, p)
    bad = [c for c in kept.columns if p in USES.get(c, set())]
    if bad:
        _fail.append(f'{p}: drop_leaky left {bad}')
log.info('(b) drop_leaky leaves no column depending on the target')

_mm = mask_rows_for_multitask(train_features, train_df.target_type.values)
for p in DFT_PROPS:
    m = (train_df.target_type == p).values
    if m.sum() and np.allclose(_mm.loc[m, f'true_{p}'], train_df.loc[m, 'target'], atol=1e-5):
        _fail.append(f'{p}: row-mask did not neutralise true_{p}')
log.info('(c) row-mask neutralises own-target columns for shared-trunk models')

log.info('(d) partner availability, train vs test (a large gap would break CV transfer):')
for p in DFT_PROPS:
    mtr = (train_df.target_type == p).values
    mte = (test_df.target_type == p).values
    if not (mtr.sum() and mte.sum()):
        continue
    for q in DFT_PROPS:
        if q == p:
            continue
        a = train_features.loc[mtr, f'true_{q}_ok'].mean()
        b = test_features.loc[mte, f'true_{q}_ok'].mean()
        if abs(a - b) > 0.25:
            log.warn(f'    {p}<-{q}: train {a:.2f} vs test {b:.2f}  (large gap)')
    cols = [f'true_{q}_ok' for q in DFT_PROPS if q != p]
    log.info(f'    {p:<4} mean partners  train {train_features.loc[mtr, cols].sum(1).mean():.2f}'
             f'  test {test_features.loc[mte, cols].sum(1).mean():.2f}')

assert not _fail, 'LEAKAGE CHECK FAILED:\n' + '\n'.join(_fail)
log.ok('all leakage checks passed')

## 7. Cross-validation folds

Folds are grouped on `poly_key`, so no molecule can appear in training and validation at the same
time — including a molecule written two different ways, which a plain `KFold` would treat as two
rows and split across the boundary. Every model in the notebook uses **the same fold assignment**,
which is what makes the stacking layer's out-of-fold matrix honest.

In [ ]:
def _make_folds():
    fold = np.full(len(train_df), -1, dtype=np.int64)
    for tt in TARGET_TYPES:
        m = np.where((train_df.target_type == tt).values)[0]
        if len(m) == 0:
            continue
        g = train_df.pkey.values[m]
        k = min(N_FOLDS, len(np.unique(g)))
        if k < 2:
            fold[m] = 0
            continue
        gk = GroupKFold(n_splits=k)
        for f, (_, vi) in enumerate(gk.split(m, groups=g)):
            fold[m[vi]] = f
        log.info(f'  {tt:<4} {len(m):>5} rows, {len(np.unique(g)):>5} groups, {k} folds')
    assert (fold >= 0).all(), 'unassigned rows in fold map'
    # a group must never straddle a fold within a property
    chk = pd.DataFrame({'p': train_df.target_type, 'g': train_df.pkey, 'f': fold})
    bad = chk.groupby(['p', 'g']).f.nunique()
    assert (bad == 1).all(), f'{(bad>1).sum()} groups straddle folds'
    log.ok('grouped folds verified: no poly_key spans two folds within a property')
    return fold


FOLD = run_stage('09_folds', _make_folds)


def prop_folds(tt):
    """(row indices, fold ids) for one property."""
    m = np.where((train_df.target_type == tt).values)[0]
    return m, FOLD[m]


def iter_folds(tt):
    m, f = prop_folds(tt)
    for k in sorted(np.unique(f)):
        yield int(k), np.where(f != k)[0], np.where(f == k)[0]

## 8. Gradient boosting — LightGBM, XGBoost, CatBoost

One model per property per fold. Each fold writes its own checkpoint holding the validation slice,
the test prediction and the fitted iteration count, so an interrupted run resumes at the exact
fold it died on rather than restarting the property. Models are not retained after their
predictions are recorded, which keeps memory flat across 210 fits.

LightGBM additionally predicts every train and test row for **every** property, not just the rows
of that property. Those cross-property predictions feed the predicted-physics step in section 12,
where a strong `egc` prediction is routed through `ei = egc + eea` to reach rows that have no
`ei`-adjacent label at all.

In [ ]:
LGBM_BASE = dict(objective='regression', metric='rmse', boosting_type='gbdt',
                 n_estimators=3000, learning_rate=0.015, max_depth=7, num_leaves=63,
                 min_child_samples=10, reg_alpha=0.1, reg_lambda=1.0,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.6,
                 random_state=SEED, verbose=-1, n_jobs=-1)

XGB_BASE = dict(objective='reg:squarederror', n_estimators=3000, learning_rate=0.015,
                max_depth=7, subsample=0.8, colsample_bytree=0.6,
                reg_alpha=0.1, reg_lambda=1.0, min_child_weight=10,
                random_state=SEED, verbosity=0)
if torch.cuda.is_available():
    XGB_BASE['device'] = 'cuda'; XGB_BASE['tree_method'] = 'hist'

CB_BASE = dict(iterations=3000, learning_rate=0.03, depth=7, l2_leaf_reg=3.0,
               random_seed=SEED, verbose=0, od_type='Iter', od_wait=100,
               allow_writing_files=False)
if torch.cuda.is_available():
    # use every visible GPU (Kaggle's T4 x2 offers two); _fit_one falls back to CPU on failure
    CB_BASE['task_type'] = 'GPU'
    CB_BASE['devices'] = '0' if torch.cuda.device_count() < 2 else f'0-{torch.cuda.device_count()-1}'

GBDT_KINDS = ['lgbm'] + (['xgb'] if HAVE_XGB else []) + (['cb'] if HAVE_CB else [])
if not HAVE_XGB:
    log.warn('xgboost unavailable - it will be omitted from the ensemble')
if not HAVE_CB:
    log.warn('catboost unavailable - it will be omitted from the ensemble')


GPU_FALLEN_BACK = set()


def _fit_one(kind, Xa, ya, Xb, yb):
    """Fit one fold, retrying on CPU if a GPU-specific backend fails.

    The XGBoost/CatBoost GPU settings are only active on Kaggle and cannot be exercised without a
    GPU, so a failure there would otherwise surface hours into a run and lose the whole stage.
    """
    try:
        return _fit_one_impl(kind, Xa, ya, Xb, yb)
    except Exception as e:
        if kind in ('xgb', 'cb') and kind not in GPU_FALLEN_BACK and torch.cuda.is_available():
            GPU_FALLEN_BACK.add(kind)
            log.warn(f'{kind} GPU backend failed ({type(e).__name__}: {e}) - falling back to CPU '
                     f'for the rest of the run')
            if kind == 'xgb':
                XGB_BASE.pop('device', None); XGB_BASE.pop('tree_method', None)
            else:
                CB_BASE.pop('task_type', None); CB_BASE.pop('devices', None)
            return _fit_one_impl(kind, Xa, ya, Xb, yb)
        raise


def _fit_one_impl(kind, Xa, ya, Xb, yb):
    if kind == 'lgbm':
        m = lgb.LGBMRegressor(**LGBM_BASE)
        m.fit(Xa, ya, eval_set=[(Xb, yb)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        return m, int(getattr(m, 'best_iteration_', 0) or LGBM_BASE['n_estimators'])
    if kind == 'xgb':
        m = xgb.XGBRegressor(early_stopping_rounds=100, **XGB_BASE)
        m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
        return m, int(getattr(m, 'best_iteration', 0) or XGB_BASE['n_estimators'])
    m = cb.CatBoostRegressor(**CB_BASE)
    m.fit(Xa.values, ya, eval_set=(Xb.values, yb), verbose=0)
    return m, int(m.get_best_iteration() or CB_BASE['iterations'])


def _predict(kind, m, X):
    return m.predict(X.values if kind == 'cb' else X)


def _run_gbdt():
    oof = {k: np.zeros(len(train_df)) for k in GBDT_KINDS}
    tep = {k: np.zeros(len(test_df)) for k in GBDT_KINDS}
    r2 = {k: {} for k in GBDT_KINDS}
    cross_tr = {q: np.zeros(len(train_df)) for q in DFT_PROPS}     # lgbm, all rows
    cross_te = {q: np.zeros(len(test_df)) for q in DFT_PROPS}

    for kind in GBDT_KINDS:
        log.sub(kind)
        for tt in TARGET_TYPES:
            rows, _ = prop_folds(tt)
            if len(rows) == 0:
                continue
            Xp = drop_leaky(train_features.iloc[rows].reset_index(drop=True), tt)
            yp = train_df.target.values[rows]
            mte = (test_df.target_type == tt).values
            Xt = drop_leaky(test_features[mte], tt)
            Xfull_tr = drop_leaky(train_features, tt) if kind == 'lgbm' and tt in DFT_PROPS else None
            Xfull_te = drop_leaky(test_features, tt) if kind == 'lgbm' and tt in DFT_PROPS else None

            oof_p = np.zeros(len(rows))
            te_acc = np.zeros(int(mte.sum())); n_fold = 0
            ctr_acc = np.zeros(len(train_df)); cte_acc = np.zeros(len(test_df))
            for f, ai, bi in iter_folds(tt):
                tag = f'gbdt/{kind}/{tt}/f{f}'
                if fhas(tag):
                    d = fload(tag)
                else:
                    t0 = time.time()
                    m, best = _fit_one(kind, Xp.iloc[ai], yp[ai], Xp.iloc[bi], yp[bi])
                    d = dict(val_idx=bi, val_pred=_predict(kind, m, Xp.iloc[bi]),
                             test_pred=_predict(kind, m, Xt), best_iter=best,
                             seconds=round(time.time() - t0, 1))
                    if Xfull_tr is not None:
                        d['cross_tr'] = _predict(kind, m, Xfull_tr)
                        d['cross_te'] = _predict(kind, m, Xfull_te)
                    fsave(tag, d)
                    log.info(f'  {kind} {tt} fold {f}: {d["seconds"]}s, {best} trees')
                oof_p[d['val_idx']] = d['val_pred']
                te_acc += d['test_pred']; n_fold += 1
                if 'cross_tr' in d:
                    ctr_acc += d['cross_tr']; cte_acc += d['cross_te']
            oof[kind][rows] = oof_p
            tep[kind][mte] = te_acc / max(n_fold, 1)
            r2[kind][tt] = r2_score(yp, oof_p)
            if tt in DFT_PROPS and kind == 'lgbm' and n_fold:
                cross_tr[tt] = ctr_acc / n_fold
                cross_te[tt] = cte_acc / n_fold
            log.metric(f'  [{tt}] {kind} grouped-OOF R2 = {r2[kind][tt]:.4f}')
        log.metric(f'>>> {kind} mean OOF R2 = {np.mean(list(r2[kind].values())):.4f}')

    return dict(oof=oof, test=tep, r2=r2, cross_tr=cross_tr, cross_te=cross_te)


GB = run_stage('10_gbdt', _run_gbdt)
tree_oof, tree_test, tree_r2 = GB['oof'], GB['test'], GB['r2']
CROSS_TR, CROSS_TE = GB['cross_tr'], GB['cross_te']

### Reference models for explanation and audit

One compact LightGBM per property, fitted on all of that property's rows. The fold models are
discarded after their predictions are recorded — keeping 210 of them would cost gigabytes — so
these stand in for SHAP attribution and for the end-to-end invariance audit. They are never used
to produce a submission value, so fitting them on all rows introduces no leakage into the score.

In [ ]:
def _reference_models():
    ref = {}
    par = dict(LGBM_BASE); par.update(n_estimators=500, learning_rate=0.05)
    for tt in TARGET_TYPES:
        rows, _ = prop_folds(tt)
        if len(rows) == 0:
            continue
        X = drop_leaky(train_features.iloc[rows].reset_index(drop=True), tt)
        m = lgb.LGBMRegressor(**par)
        m.fit(X, train_df.target.values[rows])
        ref[tt] = m
        log.info(f'  reference model [{tt}]: {X.shape[1]} features, {len(rows)} rows')
    return ref


REF_MODELS = run_stage('10b_reference_models', _reference_models)

## 9. Multi-task neural network on the shared feature matrix

One trunk, seven heads. It sees every property at once, which is how a 221-row target borrows
structure from the 4,143-row one. Own-target columns are neutralised **per row** rather than
dropped, because a column that leaks on `egc` rows is legitimate on `eps` rows.

Averaged over several seeds. Each `(seed, fold)` pair checkpoints **after every epoch** with
optimiser, scheduler and early-stopping state, so an interruption costs at most one epoch.

In [ ]:
NN_CFG = dict(hidden=[1024, 512, 256, 128], head=64, dropout=0.3, lr=1e-3,
              wd=1e-4, epochs=200, bs=64, patience=25,
              seeds=[42, 202, 777, 1337, 2024][:max(1, NN_SEED_COUNT)])


class MultiTaskNet(nn.Module):
    def __init__(self, d_in, hidden, head, n_tasks=7, p=0.3):
        super().__init__()
        L, prev = [], d_in
        for i, h in enumerate(hidden):
            L += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(),
                  nn.Dropout(max(p * (1 - i * 0.1), 0.05))]
            prev = h
        self.trunk = nn.Sequential(*L)
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(prev, head), nn.SiLU(), nn.Dropout(p * 0.3), nn.Linear(head, 1))
            for _ in range(n_tasks)])

    def forward(self, x, t):
        z = self.trunk(x)
        out = torch.zeros(x.size(0), device=x.device, dtype=z.dtype)
        for i, h in enumerate(self.heads):
            m = (t == i)
            if m.any():
                out[m] = h(z[m]).squeeze(-1).to(out.dtype)
        return out


class TabDS(Dataset):
    def __init__(self, X, y, t):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)
        self.t = torch.as_tensor(t, dtype=torch.long)

    def __len__(self): return len(self.X)

    def __getitem__(self, i): return self.X[i], self.y[i], self.t[i]


def _target_scalers(y, t, idx):
    sc = {}
    ys = y.copy().astype(np.float32)
    for tt, i in TASK_MAP.items():
        m = idx[t[idx] == i]
        s = StandardScaler()
        if len(m):
            s.fit(y[m].reshape(-1, 1))
            ys[m] = s.transform(y[m].reshape(-1, 1)).ravel()
        sc[i] = s
    return sc, ys


def _run_nn():
    Xtr = mask_rows_for_multitask(train_features, train_df.target_type.values).values
    Xte = mask_rows_for_multitask(test_features, test_df.target_type.values).values
    Xtr = np.nan_to_num(np.clip(Xtr, -3.4e38, 3.4e38)).astype(np.float32)
    Xte = np.nan_to_num(np.clip(Xte, -3.4e38, 3.4e38)).astype(np.float32)
    log.ok(f'NN inputs row-masked  {Xtr.shape}')

    oof_sum = np.zeros(len(train_df)); test_sum = np.zeros(len(test_df)); n_seed = 0
    t_begin = time.time()
    for si, sd in enumerate(NN_CFG['seeds']):
        if si > 0 and (time.time() - t_begin > BUDGET['nn'] or time_left() < 90 * 60):
            log.warn(f'NN budget reached - using {si} of {len(NN_CFG["seeds"])} seeds')
            break
        oof_s = np.zeros(len(train_df)); test_s = np.zeros(len(test_df)); nf = 0
        for f in range(N_FOLDS):
            va = np.where(FOLD == f)[0]
            trn = np.where(FOLD != f)[0]
            if len(va) == 0:
                continue
            tag = f'nn/s{sd}/f{f}'
            done_tag = tag + '/done'
            if fhas(done_tag):
                d = fload(done_tag)
                oof_s[d['val_idx']] = d['val_pred']; test_s += d['test_pred']; nf += 1
                continue

            torch.manual_seed(sd + f); np.random.seed(sd + f)
            sc = StandardScaler().fit(Xtr[trn])
            Xa = np.nan_to_num(sc.transform(Xtr[trn])).astype(np.float32)
            Xb = np.nan_to_num(sc.transform(Xtr[va])).astype(np.float32)
            tsc, ys = _target_scalers(y_all, t_all, trn)
            yb = y_all[va].astype(np.float32).copy()
            for tt, i in TASK_MAP.items():
                m = va[t_all[va] == i]
                if len(m):
                    yb[t_all[va] == i] = tsc[i].transform(y_all[m].reshape(-1, 1)).ravel()

            model = MultiTaskNet(Xa.shape[1], NN_CFG['hidden'], NN_CFG['head'],
                                 p=NN_CFG['dropout']).to(DEVICE)
            opt = torch.optim.AdamW(model.parameters(), lr=NN_CFG['lr'], weight_decay=NN_CFG['wd'])
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NN_CFG['epochs'], eta_min=1e-6)
            start_ep, best, best_state, pat = 0, 1e18, None, 0
            if fhas(tag):
                st = fload(tag)
                model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
                sch.load_state_dict(st['sch'])
                start_ep, best, best_state, pat = st['epoch'], st['best'], st['best_state'], st['pat']
                log.info(f'  resumed nn seed {sd} fold {f} at epoch {start_ep}')

            ep = start_ep - 1        # defined even when the epoch loop body never runs (fully-trained fold resumed)
            dl = DataLoader(TabDS(Xa, ys[trn], t_all[trn]), batch_size=NN_CFG['bs'],
                            shuffle=True, drop_last=True)
            vdl = DataLoader(TabDS(Xb, yb, t_all[va]), batch_size=NN_CFG['bs'] * 4)
            t0 = time.time()
            for ep in range(start_ep, NN_CFG['epochs']):
                torch.manual_seed(SEED * 7919 + sd * 131 + f * 17 + ep)   # keyed on position,
                model.train()                                            # not on run history
                for xb, yy, tb in dl:
                    xb, yy, tb = xb.to(DEVICE), yy.to(DEVICE), tb.to(DEVICE)
                    opt.zero_grad(set_to_none=True)
                    loss = F.huber_loss(model(xb, tb), yy, delta=1.0)
                    if not torch.isfinite(loss):
                        continue
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                model.eval(); vl, nv = 0.0, 0
                with torch.no_grad():
                    for xb, yy, tb in vdl:
                        xb, yy, tb = xb.to(DEVICE), yy.to(DEVICE), tb.to(DEVICE)
                        vl += F.mse_loss(model(xb, tb), yy).item() * len(xb); nv += len(xb)
                vl /= max(nv, 1); sch.step()
                if vl < best - 1e-6:
                    best, best_state, pat = vl, cpu_state(model.state_dict()), 0
                else:
                    pat += 1
                fsave(tag, dict(model=cpu_state(model.state_dict()), opt=opt.state_dict(),
                                sch=sch.state_dict(), epoch=ep + 1, best=best,
                                best_state=best_state, pat=pat))
                if pat >= NN_CFG['patience']:
                    break

            if best_state:
                model.load_state_dict(best_state)
            model.eval()

            def _inv(pred, tsel):
                out = np.zeros_like(pred)
                for tt, i in TASK_MAP.items():
                    m = (tsel == i)
                    if m.any():
                        out[m] = tsc[i].inverse_transform(pred[m].reshape(-1, 1)).ravel()
                return out

            with torch.no_grad():
                pv = model(torch.as_tensor(Xb, device=DEVICE),
                           torch.as_tensor(t_all[va], device=DEVICE)).cpu().numpy()
                Xs = np.nan_to_num(sc.transform(Xte)).astype(np.float32)
                pt = np.zeros(len(Xs))
                for i in range(0, len(Xs), 1024):
                    j = min(i + 1024, len(Xs))
                    pt[i:j] = model(torch.as_tensor(Xs[i:j], device=DEVICE),
                                    torch.as_tensor(t_test[i:j], device=DEVICE)).cpu().numpy()
            d = dict(val_idx=va, val_pred=_inv(pv, t_all[va]), test_pred=_inv(pt, t_test),
                     epochs=ep + 1, seconds=round(time.time() - t0, 1))
            fsave(done_tag, d)
            oof_s[va] = d['val_pred']; test_s += d['test_pred']; nf += 1
            log.metric(f'  nn seed {sd} fold {f}: {d["epochs"]} epochs, {d["seconds"]}s')

        if nf:
            oof_sum += oof_s; test_sum += test_s / nf; n_seed += 1
            r = np.mean([r2_score(y_all[t_all == i], oof_s[t_all == i]) for i in TASK_MAP.values()])
            log.metric(f'  >> seed {sd} mean OOF R2 = {r:.4f}')

    n_seed = max(n_seed, 1)
    oof = oof_sum / n_seed; tst = test_sum / n_seed
    r2 = {tt: r2_score(y_all[t_all == i], oof[t_all == i]) for tt, i in TASK_MAP.items()}
    for tt in TARGET_TYPES:
        log.metric(f'  NN [{tt}] R2 = {r2[tt]:.4f}')
    log.metric(f'>>> NN mean OOF R2 = {np.mean(list(r2.values())):.4f}')
    return dict(oof=oof, test=tst, r2=r2, n_seed=n_seed)


NNR = run_stage('11_multitask_nn', _run_nn)

## 10. Fine-tuning the PI1M encoder

The pretrained encoder gets seven regression heads on its mean-pooled representation and is
fine-tuned on all properties jointly, with a lower learning rate for the pretrained trunk than
for the fresh heads.

**Invariance is trained in, not just normalised in.** Each training row is presented as several
valid spellings of the same polymer — the canonical form, its dimer, and randomly re-ordered
SMILES — all carrying the identical label. The model is therefore penalised for letting the
prediction move when only the spelling moves. At inference the same variants are averaged (TTA),
which converts any residual sensitivity into variance reduction. Section 14 measures what is
left.

Checkpointed per epoch; the wall-clock cap reduces the fold count rather than truncating training.

In [ ]:
FT_CFG = dict(lr_head=3e-4, lr_enc=5e-5, wd=0.01, epochs=30, bs=64, patience=6,
              n_rand=2, oligo=(2,), tta_rand=2, tta_oligo=(2,), warmup_frac=0.1)


class PolymerRegressor(nn.Module):
    def __init__(self, encoder, d, n_tasks=7, p=0.15):
        super().__init__()
        self.enc = encoder
        self.drop = nn.Dropout(p)
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Dropout(p), nn.Linear(128, 1))
            for _ in range(n_tasks)])

    def represent(self, x):
        return self.enc.pool(x)

    def forward(self, x, t):
        z = self.drop(self.represent(x))
        out = torch.zeros(x.size(0), device=x.device, dtype=z.dtype)
        for i, h in enumerate(self.heads):
            m = (t == i)
            if m.any():
                # .to(out.dtype): under autocast a Linear emits half while pool() returns float32,
                # because its mean divides by a float mask. Scattering half into float raises.
                out[m] = h(z[m]).squeeze(-1).to(out.dtype)
        return out


def _variant_tokens(smiles, n_rand, oligo, seed=SEED):
    """Token ids for several equivalent spellings of each molecule: [n_mol, n_var, MAX_LEN].

    Seeded per molecule, so the same molecule always yields the same variants regardless of where
    it sits in the array or whether the run was resumed.
    """
    n_var = 1 + len(oligo) + n_rand
    out = np.zeros((len(smiles), n_var, MAX_LEN), dtype=np.int64)
    cache = {}
    for i, s in enumerate(smiles):
        if s not in cache:
            v = [s] + [make_oligomer(s, k) for k in oligo]
            mol_seed = seed + int(hashlib.md5(s.encode()).hexdigest()[:8], 16)
            v += rand_smiles_many(s, n_rand, mol_seed)
            v = (v + [s] * n_var)[:n_var]
            cache[s] = np.stack([encode(x, VOCAB) for x in v])
        out[i] = cache[s]
    return out


def _run_finetune():
    if not USE_NEURAL or pretrained.get('state') is None:
        log.warn('neural stages disabled - the transformer is excluded from the ensemble')
        return None
    log.info('building representation variants for train and test...')
    t0 = time.time()
    TRV = _variant_tokens(train_df.nsmiles.values, FT_CFG['n_rand'], FT_CFG['oligo'])
    TEV = _variant_tokens(test_df.nsmiles.values, FT_CFG['tta_rand'], FT_CFG['tta_oligo'])
    log.info(f'  train variants {TRV.shape}, test variants {TEV.shape} ({time.time()-t0:.0f}s)')

    oof = np.zeros(len(train_df)); test_acc = np.zeros(len(test_df)); nfold = 0
    t_begin = time.time()
    for f in range(N_FOLDS):
        va = np.where(FOLD == f)[0]
        trn = np.where(FOLD != f)[0]
        if len(va) == 0:
            continue
        done_tag = f'ft/f{f}/done'
        if fhas(done_tag):
            d = fload(done_tag)
            oof[d['val_idx']] = d['val_pred']; test_acc += d['test_pred']; nfold += 1
            log.ok(f'  fine-tune fold {f}: restored')
            continue
        if nfold > 0 and (time.time() - t_begin > BUDGET['finetune'] or time_left() < 55 * 60):
            log.warn(f'fine-tune budget reached - {nfold} of {N_FOLDS} folds trained')
            break

        torch.manual_seed(SEED + f); np.random.seed(SEED + f)
        model = PolymerRegressor(_encoder_from(pretrained['state']), PRE_CFG['d_model']).to(DEVICE)
        enc_p = list(model.enc.parameters())
        head_p = [p for n, p in model.named_parameters() if not n.startswith('enc.')]
        opt = torch.optim.AdamW([{'params': enc_p, 'lr': FT_CFG['lr_enc']},
                                 {'params': head_p, 'lr': FT_CFG['lr_head']}],
                                weight_decay=FT_CFG['wd'])

        tsc, ys = _target_scalers(y_all, t_all, trn)
        yb = y_all[va].astype(np.float32).copy()
        for tt, i in TASK_MAP.items():
            sel = t_all[va] == i
            if sel.any():
                yb[sel] = tsc[i].transform(y_all[va][sel].reshape(-1, 1)).ravel()

        n_var = TRV.shape[1]
        flat_x = TRV[trn].reshape(-1, MAX_LEN)
        flat_y = np.repeat(ys[trn], n_var).astype(np.float32)
        flat_t = np.repeat(t_all[trn], n_var)
        steps_ep = max(1, len(flat_x) // FT_CFG['bs'])
        total = steps_ep * FT_CFG['epochs']
        warm = int(total * FT_CFG['warmup_frac'])

        def lr_at(s):
            if s < warm:
                return s / max(warm, 1)
            return 0.5 * (1 + math.cos(math.pi * min((s - warm) / max(total - warm, 1), 1.0)))

        sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
        autocast, scaler = _amp()
        start_ep, best, best_state, pat = 0, 1e18, None, 0
        tag = f'ft/f{f}'
        if fhas(tag):
            st = fload(tag)
            model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
            sch.load_state_dict(st['sch'])
            if scaler is not None and st.get('scaler'):
                scaler.load_state_dict(st['scaler'])
            start_ep, best, best_state, pat = st['epoch'], st['best'], st['best_state'], st['pat']
            log.info(f'  resumed fine-tune fold {f} at epoch {start_ep}')

        ep = start_ep - 1        # defined even when the epoch loop body never runs (fully-trained fold resumed)
        val_x = torch.as_tensor(TRV[va][:, 0, :], device=DEVICE)
        val_t = torch.as_tensor(t_all[va], device=DEVICE)
        val_y = torch.as_tensor(yb, device=DEVICE)
        t0 = time.time()
        for ep in range(start_ep, FT_CFG['epochs']):
            torch.manual_seed(SEED * 6151 + f * 17 + ep)                  # keyed on position,
            model.train()                                                 # not on run history
            order = np.random.default_rng(SEED * 6151 + f * 17 + ep).permutation(len(flat_x))
            for bi in range(steps_ep):
                sl = order[bi * FT_CFG['bs']:(bi + 1) * FT_CFG['bs']]
                if len(sl) < 2:
                    continue
                xb = torch.as_tensor(flat_x[sl], device=DEVICE)
                yy = torch.as_tensor(flat_y[sl], device=DEVICE)
                tb = torch.as_tensor(flat_t[sl], device=DEVICE)
                opt.zero_grad(set_to_none=True)
                with autocast:
                    loss = F.huber_loss(model(xb, tb), yy, delta=1.0)
                if not torch.isfinite(loss):
                    continue
                if scaler is not None:
                    scaler.scale(loss).backward(); scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                sch.step()
            model.eval(); vl = 0.0
            with torch.no_grad():
                for i in range(0, len(val_x), 512):
                    j = min(i + 512, len(val_x))
                    vl += F.mse_loss(model(val_x[i:j], val_t[i:j]), val_y[i:j]).item() * (j - i)
            vl /= max(len(val_x), 1)
            if vl < best - 1e-6:
                best, best_state, pat = vl, cpu_state(model.state_dict()), 0
            else:
                pat += 1
            fsave(tag, dict(model=cpu_state(model.state_dict()), opt=opt.state_dict(),
                            sch=sch.state_dict(),
                            scaler=scaler.state_dict() if scaler is not None else None,
                            epoch=ep + 1, best=best, best_state=best_state, pat=pat))
            if (ep + 1) % 5 == 0 or pat >= FT_CFG['patience']:
                log.metric(f'  ft fold {f} epoch {ep+1}: val {vl:.4f} best {best:.4f} pat {pat}')
            if pat >= FT_CFG['patience']:
                break

        if best_state:
            model.load_state_dict(best_state)
        model.eval()

        def _inv(pred, tsel):
            out = np.zeros_like(pred)
            for tt, i in TASK_MAP.items():
                m = (tsel == i)
                if m.any():
                    out[m] = tsc[i].inverse_transform(pred[m].reshape(-1, 1)).ravel()
            return out

        def _tta(tok3, tsel):
            """Average the prediction over every equivalent spelling."""
            acc = np.zeros(tok3.shape[0])
            with torch.no_grad():
                for v in range(tok3.shape[1]):
                    p = np.zeros(tok3.shape[0])
                    for i in range(0, tok3.shape[0], 512):
                        j = min(i + 512, tok3.shape[0])
                        p[i:j] = model(torch.as_tensor(tok3[i:j, v, :], device=DEVICE),
                                       torch.as_tensor(tsel[i:j], device=DEVICE)).float().cpu().numpy()
                    acc += p
            return acc / tok3.shape[1]

        d = dict(val_idx=va, val_pred=_inv(_tta(TRV[va], t_all[va]), t_all[va]),
                 test_pred=_inv(_tta(TEV, t_test), t_test),
                 epochs=ep + 1, seconds=round(time.time() - t0, 1))
        fsave(done_tag, d)
        oof[va] = d['val_pred']; test_acc += d['test_pred']; nfold += 1
        log.metric(f'  fine-tune fold {f}: {d["epochs"]} epochs, {d["seconds"]/60:.1f} min')
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if nfold == 0:
        log.warn('no fine-tune fold completed - the transformer is excluded from the ensemble')
        return None
    if nfold < N_FOLDS:
        miss = np.where(oof == 0)[0]
        if len(miss):
            for tt, i in TASK_MAP.items():
                sel = miss[t_all[miss] == i]
                if len(sel):
                    seen = oof[(t_all == i) & (oof != 0)]
                    oof[sel] = seen.mean() if len(seen) else y_all[t_all == i].mean()
            log.warn(f'{len(miss)} rows had no fine-tune fold; filled with the property mean')
    test = test_acc / nfold
    r2 = {tt: r2_score(y_all[t_all == i], oof[t_all == i]) for tt, i in TASK_MAP.items()}
    for tt in TARGET_TYPES:
        log.metric(f'  FT [{tt}] R2 = {r2[tt]:.4f}')
    log.metric(f'>>> fine-tuned encoder mean OOF R2 = {np.mean(list(r2.values())):.4f}')
    return dict(oof=oof, test=test, r2=r2, folds=nfold)


FT = run_stage('12_finetune', _run_finetune)

## 11. SMILES 1D-CNN

A convolutional character model, kept for ensemble diversity: it makes different mistakes from
both the descriptor trees and the transformer. It reads SMILES only, so it needs no leakage guard.
Trained on augmented spellings and averaged over spellings at inference, like the transformer.
Skipped automatically if the budget is already spent.

In [ ]:
CNN_CFG = dict(embed=64, filters=128, kernels=[3, 5, 7, 11], fc=256, dropout=0.3,
               lr=5e-4, wd=1e-4, epochs=100, bs=64, patience=15, n_var=4)


class SmilesCNN(nn.Module):
    def __init__(self, vocab, p=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab, CNN_CFG['embed'], padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Sequential(
            nn.Conv1d(CNN_CFG['embed'], CNN_CFG['filters'], k, padding=k // 2),
            nn.BatchNorm1d(CNN_CFG['filters']), nn.SiLU()) for k in CNN_CFG['kernels']])
        d = CNN_CFG['filters'] * len(CNN_CFG['kernels']) * 2
        self.fc = nn.Sequential(nn.Linear(d, CNN_CFG['fc']), nn.BatchNorm1d(CNN_CFG['fc']),
                                nn.SiLU(), nn.Dropout(p))
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(CNN_CFG['fc'], 64), nn.SiLU(), nn.Dropout(p * 0.3),
                          nn.Linear(64, 1)) for _ in range(7)])

    def feats(self, x):
        e = self.emb(x).transpose(1, 2)
        o = []
        for c in self.convs:
            z = c(e); o.append(z.mean(2)); o.append(z.max(2).values)
        return self.fc(torch.cat(o, 1))

    def forward(self, x, t):
        h = self.feats(x)
        out = torch.zeros(x.size(0), device=x.device, dtype=h.dtype)
        for i, hd in enumerate(self.heads):
            m = (t == i)
            if m.any():
                out[m] = hd(h[m]).squeeze(-1).to(out.dtype)
        return out


def _run_cnn():
    if not USE_NEURAL:
        log.warn('neural stages disabled - the CNN is excluded from the ensemble')
        return None
    if time_left() < 70 * 60:
        log.warn('not enough budget left for the CNN - skipping it')
        return None
    TRV = _variant_tokens(train_df.nsmiles.values, CNN_CFG['n_var'] - 2, (2,), seed=SEED + 7)
    TEV = _variant_tokens(test_df.nsmiles.values, 1, (2,), seed=SEED + 7)
    oof = np.zeros(len(train_df)); test_acc = np.zeros(len(test_df)); nfold = 0
    t_begin = time.time()
    for f in range(N_FOLDS):
        va = np.where(FOLD == f)[0]; trn = np.where(FOLD != f)[0]
        if len(va) == 0:
            continue
        done_tag = f'cnn/f{f}/done'
        if fhas(done_tag):
            d = fload(done_tag)
            oof[d['val_idx']] = d['val_pred']; test_acc += d['test_pred']; nfold += 1
            continue
        if nfold > 0 and (time.time() - t_begin > BUDGET['cnn'] or time_left() < 40 * 60):
            log.warn(f'CNN budget reached - {nfold} of {N_FOLDS} folds trained')
            break
        torch.manual_seed(SEED + f); np.random.seed(SEED + f)
        model = SmilesCNN(V, CNN_CFG['dropout']).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=CNN_CFG['lr'], weight_decay=CNN_CFG['wd'])
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNN_CFG['epochs'], eta_min=1e-6)
        tsc, ys = _target_scalers(y_all, t_all, trn)
        yb = y_all[va].astype(np.float32).copy()
        for tt, i in TASK_MAP.items():
            sel = t_all[va] == i
            if sel.any():
                yb[sel] = tsc[i].transform(y_all[va][sel].reshape(-1, 1)).ravel()
        nv = TRV.shape[1]
        fx = TRV[trn].reshape(-1, MAX_LEN)
        fy = np.repeat(ys[trn], nv).astype(np.float32)
        ft = np.repeat(t_all[trn], nv)
        steps = max(1, len(fx) // CNN_CFG['bs'])
        vx = torch.as_tensor(TRV[va][:, 0, :], device=DEVICE)
        vt = torch.as_tensor(t_all[va], device=DEVICE)
        vy = torch.as_tensor(yb, device=DEVICE)
        tag = f'cnn/f{f}'
        start_ep, best, best_state, pat = 0, 1e18, None, 0
        if fhas(tag):
            st = fload(tag)
            model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
            sch.load_state_dict(st['sch'])
            start_ep, best, best_state, pat = st['epoch'], st['best'], st['best_state'], st['pat']
        ep = start_ep - 1        # defined even when the epoch loop body never runs
        t0 = time.time()
        for ep in range(start_ep, CNN_CFG['epochs']):
            torch.manual_seed(SEED * 4409 + f * 17 + ep)                  # keyed on position,
            model.train()                                                 # not on run history
            order = np.random.default_rng(SEED * 4409 + f * 17 + ep).permutation(len(fx))
            for bi in range(steps):
                sl = order[bi * CNN_CFG['bs']:(bi + 1) * CNN_CFG['bs']]
                if len(sl) < 2:
                    continue
                xb = torch.as_tensor(fx[sl], device=DEVICE)
                opt.zero_grad(set_to_none=True)
                loss = F.huber_loss(model(xb, torch.as_tensor(ft[sl], device=DEVICE)),
                                    torch.as_tensor(fy[sl], device=DEVICE), delta=1.0)
                if not torch.isfinite(loss):
                    continue
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sch.step()
            model.eval(); vl = 0.0
            with torch.no_grad():
                for i in range(0, len(vx), 512):
                    j = min(i + 512, len(vx))
                    vl += F.mse_loss(model(vx[i:j], vt[i:j]), vy[i:j]).item() * (j - i)
            vl /= max(len(vx), 1)
            if vl < best - 1e-6:
                best, best_state, pat = vl, cpu_state(model.state_dict()), 0
            else:
                pat += 1
            fsave(tag, dict(model=cpu_state(model.state_dict()), opt=opt.state_dict(),
                            sch=sch.state_dict(), epoch=ep + 1, best=best,
                            best_state=best_state, pat=pat))
            if pat >= CNN_CFG['patience']:
                break
        if best_state:
            model.load_state_dict(best_state)
        model.eval()

        def _inv(pred, tsel):
            out = np.zeros_like(pred)
            for tt, i in TASK_MAP.items():
                m = (tsel == i)
                if m.any():
                    out[m] = tsc[i].inverse_transform(pred[m].reshape(-1, 1)).ravel()
            return out

        def _tta(tok3, tsel):
            acc = np.zeros(tok3.shape[0])
            with torch.no_grad():
                for v in range(tok3.shape[1]):
                    p = np.zeros(tok3.shape[0])
                    for i in range(0, tok3.shape[0], 512):
                        j = min(i + 512, tok3.shape[0])
                        p[i:j] = model(torch.as_tensor(tok3[i:j, v, :], device=DEVICE),
                                       torch.as_tensor(tsel[i:j], device=DEVICE)).float().cpu().numpy()
                    acc += p
            return acc / tok3.shape[1]

        d = dict(val_idx=va, val_pred=_inv(_tta(TRV[va], t_all[va]), t_all[va]),
                 test_pred=_inv(_tta(TEV, t_test), t_test),
                 epochs=ep + 1, seconds=round(time.time() - t0, 1))
        fsave(done_tag, d)
        oof[va] = d['val_pred']; test_acc += d['test_pred']; nfold += 1
        log.metric(f'  cnn fold {f}: {d["epochs"]} epochs, {d["seconds"]:.0f}s')
        del model; gc.collect()
    if nfold == 0:
        return None
    if nfold < N_FOLDS:
        miss = np.where(oof == 0)[0]
        for tt, i in TASK_MAP.items():
            sel = miss[t_all[miss] == i]
            if len(sel):
                seen = oof[(t_all == i) & (oof != 0)]
                oof[sel] = seen.mean() if len(seen) else y_all[t_all == i].mean()
    test = test_acc / nfold
    r2 = {tt: r2_score(y_all[t_all == i], oof[t_all == i]) for tt, i in TASK_MAP.items()}
    for tt in TARGET_TYPES:
        log.metric(f'  CNN [{tt}] R2 = {r2[tt]:.4f}')
    log.metric(f'>>> CNN mean OOF R2 = {np.mean(list(r2.values())):.4f}')
    return dict(oof=oof, test=test, r2=r2, folds=nfold)


CNNR = run_stage('13_cnn', _run_cnn)

## 12. Ridge stacking

A per-property ridge over the base models' out-of-fold predictions, with the ridge penalty chosen
by an inner grouped split. Because every base model used the same grouped folds, this second
level never sees a molecule it was fitted on.

In [ ]:
def _collect_bases():
    d = {k: (tree_oof[k], tree_test[k]) for k in GBDT_KINDS}
    d['nn'] = (NNR['oof'], NNR['test'])
    if FT is not None:
        d['ft'] = (FT['oof'], FT['test'])
    if CNNR is not None:
        d['cnn'] = (CNNR['oof'], CNNR['test'])
    return d


BASES = _collect_bases()
BASE_NAMES = sorted(BASES)
log.info(f'stacking over {len(BASE_NAMES)} base models: {BASE_NAMES}')


def _stack():
    final = np.zeros(len(test_df)); stack_r2 = {}; weights = {}
    for tt in TARGET_TYPES:
        rows, folds = prop_folds(tt)
        mte = (test_df.target_type == tt).values
        if len(rows) == 0 or mte.sum() == 0:
            continue
        mX = np.nan_to_num(np.column_stack([BASES[n][0][rows] for n in BASE_NAMES]))
        tX = np.nan_to_num(np.column_stack([BASES[n][1][mte] for n in BASE_NAMES]))
        my = train_df.target.values[rows]

        best_a, best_s = 1.0, -1e18
        for a in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
            sc_ = []
            for f in np.unique(folds):
                ti, vi = np.where(folds != f)[0], np.where(folds == f)[0]
                if len(vi) < 2 or len(ti) < 5:
                    continue
                s = StandardScaler(); r = Ridge(alpha=a, random_state=SEED)
                r.fit(np.nan_to_num(s.fit_transform(mX[ti])), my[ti])
                sc_.append(r2_score(my[vi], r.predict(np.nan_to_num(s.transform(mX[vi])))))
            if sc_ and np.mean(sc_) > best_s:
                best_s, best_a = np.mean(sc_), a

        oof_m = np.zeros(len(my))
        for f in np.unique(folds):
            ti, vi = np.where(folds != f)[0], np.where(folds == f)[0]
            if len(vi) == 0:
                continue
            s = StandardScaler(); r = Ridge(alpha=best_a, random_state=SEED)
            r.fit(np.nan_to_num(s.fit_transform(mX[ti])), my[ti])
            oof_m[vi] = r.predict(np.nan_to_num(s.transform(mX[vi])))
        stack_r2[tt] = r2_score(my, oof_m)

        s = StandardScaler(); r = Ridge(alpha=best_a, random_state=SEED)
        r.fit(np.nan_to_num(s.fit_transform(mX)), my)
        final[mte] = r.predict(np.nan_to_num(s.transform(tX)))
        weights[tt] = dict(zip(BASE_NAMES, np.round(r.coef_, 4).tolist()))
        log.metric(f'  [{tt}] alpha={best_a:<7g} stack OOF R2={stack_r2[tt]:.4f}  '
                   f'weights={weights[tt]}')

    log.metric(f'>>> STACK MEAN OOF R2 = {np.mean(list(stack_r2.values())):.4f}')
    return dict(final=final, r2=stack_r2, weights=weights, oof_names=BASE_NAMES)


STACK = run_stage('14_stack', _stack)
final = STACK['final'].copy()
stack_r2 = STACK['r2']

# ---- per-model comparison table ----
log.set_stage('14_stack')
_hdr = f'{"target":<8}{"n":>6}' + ''.join(f'{n:>9}' for n in BASE_NAMES) + f'{"stack":>9}'
print('\n' + _hdr); print('-' * len(_hdr))
for tt in TARGET_TYPES:
    n = int((train_df.target_type == tt).sum())
    row = f'{tt:<8}{n:>6}'
    for nme in BASE_NAMES:
        rr = (tree_r2[nme][tt] if nme in GBDT_KINDS else
              (NNR['r2'][tt] if nme == 'nn' else
               (FT['r2'][tt] if nme == 'ft' else CNNR['r2'][tt])))
        row += f'{rr:>9.4f}'
    print(row + f'{stack_r2.get(tt, float("nan")):>9.4f}')
print('-' * len(_hdr))


def _mean_of(nme):
    d = (tree_r2[nme] if nme in GBDT_KINDS else
         (NNR['r2'] if nme == 'nn' else (FT['r2'] if nme == 'ft' else CNNR['r2'])))
    return np.mean(list(d.values()))


print(f'{"MEAN":<8}{"":>6}' + ''.join(f'{_mean_of(n):>9.4f}' for n in BASE_NAMES)
      + f'{np.mean(list(stack_r2.values())):>9.4f}')

## 13. Physics blending

The physics identities were already handed over as features, but an axis-aligned tree can only
approximate `ei = egc + eea` with staircases, and as one column among ~5,700 inside a six-model
stack it gets badly under-weighted. So the relation is applied a second time, directly.

Two passes, over disjoint row sets:

1. **Covered rows** — the true partner labels exist in train. The estimator is calibrated
   out-of-fold on train rows, the blend weight is chosen on those out-of-fold values, then both
   are applied to test.
2. **Uncovered rows** — no true partner, so the partner is supplied by LightGBM's own
   cross-property prediction. A row is uncovered precisely because that molecule carries no label
   for the partner property, so it was never in that property's training set and the prediction
   is out-of-sample.

Both weights are shrunk toward the model, because they are selected on as few as 59 rows. If
physics does not help a property the weight comes out zero and that property is left untouched.

In [ ]:
PHYS = {
    'ei':  (['egc', 'eea'], lambda d: d[:, 0] + d[:, 1]),
    'eea': (['ei', 'egc'],  lambda d: d[:, 0] - d[:, 1]),
    'egb': (['egc'],        lambda d: d[:, 0]),
    'eps': (['nc'],         lambda d: d[:, 0] ** 2),
    'nc':  (['eps'],        lambda d: np.sqrt(np.clip(d[:, 0], 0, None))),
}
SHRINK = 0.75


def _true_partner(df, props):
    return np.column_stack([df.pkey.map(TRUTH[q]).values.astype(np.float64) for q in props])


def _stack_oof_for(tt, rows, folds):
    mX = np.nan_to_num(np.column_stack([BASES[n][0][rows] for n in BASE_NAMES]))
    y = train_df.target.values[rows]
    out = np.zeros(len(y))
    for f in np.unique(folds):
        ti, vi = np.where(folds != f)[0], np.where(folds == f)[0]
        if len(vi) == 0:
            continue
        s = StandardScaler(); r = Ridge(alpha=1.0, random_state=SEED)
        r.fit(np.nan_to_num(s.fit_transform(mX[ti])), y[ti])
        out[vi] = r.predict(np.nan_to_num(s.transform(mX[vi])))
    return y, out


def _blend(final, mode):
    """mode='true' uses observed partners; mode='pred' uses LightGBM cross-property predictions."""
    applied, report = 0, {}
    for p, (srcs, fn) in PHYS.items():
        rows, folds = prop_folds(p)
        mte = (test_df.target_type == p).values
        if len(rows) == 0 or mte.sum() == 0:
            continue
        y, stack_oof = _stack_oof_for(p, rows, folds)

        cov_tr = np.isfinite(_true_partner(train_df.iloc[rows], srcs)).all(1)
        cov_te = np.isfinite(_true_partner(test_df[mte], srcs)).all(1)
        if mode == 'true':
            sel_tr, sel_te = cov_tr, cov_te
            D_tr = _true_partner(train_df.iloc[rows], srcs)[sel_tr]
            D_te = _true_partner(test_df[mte], srcs)[sel_te]
        else:
            sel_tr, sel_te = ~cov_tr, ~cov_te
            D_tr = np.column_stack([CROSS_TR[q][rows] for q in srcs])[sel_tr]
            D_te = np.column_stack([CROSS_TE[q][mte] for q in srcs])[sel_te]

        if sel_tr.sum() < 25 or sel_te.sum() == 0:
            log.info(f'  {p} [{mode}]: {sel_tr.sum()} train / {sel_te.sum()} test rows - skipped')
            continue

        est_tr = fn(D_tr); yc = y[sel_tr]; mc = stack_oof[sel_tr]
        if not np.isfinite(est_tr).all():
            log.info(f'  {p} [{mode}]: non-finite estimator - skipped'); continue

        cal = np.zeros(len(est_tr))
        for a, b in KFold(5, shuffle=True, random_state=SEED).split(est_tr):
            A = np.c_[est_tr[a], np.ones(len(a))]
            w_, *_ = np.linalg.lstsq(A, yc[a], rcond=None)
            cal[b] = np.c_[est_tr[b], np.ones(len(b))] @ w_

        bw, br = 0.0, -1e18
        for w in np.arange(0, 1.001, 0.05):
            r = r2_score(yc, (1 - w) * mc + w * cal)
            if r > br:
                br, bw = r, w
        w_use = SHRINK * bw
        if w_use <= 0:
            log.info(f'  {p} [{mode}]: physics adds nothing (w=0) - left untouched'); continue

        A = np.c_[est_tr, np.ones(len(est_tr))]
        coef, *_ = np.linalg.lstsq(A, yc, rcond=None)
        est_te = fn(D_te)
        cal_te = np.c_[est_te, np.ones(len(est_te))] @ coef
        idx = np.where(mte)[0][sel_te]
        before = final[idx].copy()
        final[idx] = (1 - w_use) * before + w_use * cal_te
        applied += len(idx)
        report[p] = dict(mode=mode, n_test=int(len(idx)), w=round(float(w_use), 3),
                         stack_r2=round(float(r2_score(yc, mc)), 4),
                         physics_r2=round(float(r2_score(yc, cal)), 4),
                         blend_r2=round(float(br), 4),
                         mean_shift=round(float(np.abs(final[idx] - before).mean()), 4))
        log.metric(f'  {p:4s} [{mode}] train {sel_tr.sum():4d}/{len(rows):<4d} '
                   f'test {sel_te.sum():4d}/{int(mte.sum()):<4d} | stack {report[p]["stack_r2"]:.4f} '
                   f'physics {report[p]["physics_r2"]:.4f} blend {br:.4f} | '
                   f'w={bw:.2f}->{w_use:.2f} | mean shift {report[p]["mean_shift"]:.3f}')
    log.ok(f'[{mode}] physics applied to {applied} test rows')
    assert np.isfinite(final).all(), f'physics blend [{mode}] produced non-finite values'
    return final, report


def _physics():
    f = final.copy()
    f, rep_true = _blend(f, 'true')
    f, rep_pred = _blend(f, 'pred')
    return dict(final=f, true=rep_true, pred=rep_pred)


PHYSR = run_stage('15_physics', _physics)
final = PHYSR['final'].copy()

## 14. Invariance audit — end to end

Section 1 proved the *key* is invariant. This measures the **whole inference path**: raw SMILES →
normalise → featurise → partner join → model. Test molecules are re-spelled as random SMILES and
as dimers, pushed through the real pipeline, and the spread of the resulting predictions is
reported per property, in units of that property's own standard deviation.

A control run repeats the same measurement with normalisation disabled, which is what shows the
invariance is bought by the pipeline rather than assumed.

In [ ]:
def _invariance_audit():
    lgbm_models = REF_MODELS or {}
    if not lgbm_models:
        log.warn('no retained model available for the audit - skipping')
        return None
    rng = np.random.default_rng(SEED)
    n_probe = min(200, len(test_df))
    probe = rng.choice(len(test_df), n_probe, replace=False)
    rows = test_df.iloc[probe]

    def spellings(smi):
        base = norm_smiles(smi)
        seed = SEED + int(hashlib.md5(smi.encode()).hexdigest()[:8], 16)
        return [smi, base, make_oligomer(base, 2)] + rand_smiles_many(base, 2, seed)

    var = {i: spellings(s) for i, s in zip(rows.index, rows.smiles.values)}
    n_var = len(next(iter(var.values())))

    def predict_path(smiles_list, tts, keys, normalise):
        src = [norm_smiles(s) if normalise else s for s in smiles_list]
        F = clean_features(featurize_batch(src)).astype(np.float32)
        for c in BASE_COLS:
            if c not in F.columns:
                F[c] = 0.0
        F = F[BASE_COLS]
        F = F.reset_index(drop=True)
        if pretrained.get('state') is not None:
            emb = pd.DataFrame(_embed(src, _encoder_from(pretrained['state'])),
                               columns=[f'emb_{i}' for i in range(PRE_CFG['d_model'])])
            F = pd.concat([F, emb], axis=1)
        ks = pd.Series([ring_key(reduce_repeat(s)) if normalise else ring_key(s) for s in src])
        raw = {q: ks.map(TRUTH[q]).values.astype(np.float64) for q in DFT_PROPS}
        for name, v, _ in physics_terms(raw):        # the same definitions used in training
            F[name] = np.where(np.isfinite(v), v, FILLS[name]).astype(np.float32)
            F[f'{name}_ok'] = np.isfinite(v).astype(np.float32)
        F = F[FEAT_COLS]
        out = np.zeros(len(F))
        for tt in set(tts):
            m = np.array([t == tt for t in tts])
            if tt not in lgbm_models or not m.any():
                continue
            out[m] = lgbm_models[tt].predict(drop_leaky(F[m], tt))
        return out

    flat_s, flat_t, flat_i = [], [], []
    for i in rows.index:
        for s in var[i]:
            flat_s.append(s); flat_t.append(test_df.target_type[i]); flat_i.append(i)

    res = {}
    for label, normalise in [('with normalisation', True), ('control: normalisation off', False)]:
        log.sub(label)
        p = predict_path(flat_s, flat_t, flat_i, normalise)
        P = p.reshape(n_probe, n_var)
        per = {}
        for tt in TARGET_TYPES:
            m = np.array([test_df.target_type[i] == tt for i in rows.index])
            if m.sum() < 3:
                continue
            sd = train_df.loc[train_df.target_type == tt, 'target'].std()
            spread = P[m].std(1) / max(sd, 1e-9)
            rng_ = (P[m].max(1) - P[m].min(1)) / max(sd, 1e-9)
            per[tt] = dict(n=int(m.sum()), mean_rel_sd=round(float(spread.mean()), 5),
                           max_rel_range=round(float(rng_.max()), 5))
            log.metric(f'  {tt:<4} n={m.sum():<4} mean spread {spread.mean():.5f} sd  '
                       f'worst range {rng_.max():.5f} sd')
        overall = float(np.mean([v['mean_rel_sd'] for v in per.values()])) if per else float('nan')
        log.metric(f'  >>> mean prediction spread across spellings: {overall:.5f} sd')
        res[label] = dict(per_property=per, overall_rel_sd=overall)
    a = res['with normalisation']['overall_rel_sd']
    b = res['control: normalisation off']['overall_rel_sd']
    if np.isfinite(a) and np.isfinite(b) and b > 0:
        log.ok(f'normalisation reduces representation sensitivity {b/max(a,1e-9):.1f}x '
               f'({b:.5f} -> {a:.5f} sd)')
    return res


INV_AUDIT = run_stage('16_invariance_audit', _invariance_audit)

## 15. Explainability

Three views, aimed at what a materials scientist would actually ask.

- **What drives each property?** SHAP on the per-property LightGBM, aggregated into families
  (descriptor, fingerprint, polymer topology, observed partner, physics term, learned embedding)
  so the answer is readable rather than 5,700 columns of noise, plus the individual top features.
- **Which physics carried which property?** The blend report already records, per property, how
  the unfitted relation scored against the model and how much weight it earned.
- **Where in the string is the model looking?** Input-gradient saliency over the fine-tuned
  encoder's tokens, mapped back to SMILES substrings.

In [ ]:
def _family(c):
    if c.startswith(('mfp2_', 'mfp3_', 'ap_', 'tt_', 'mac_')):
        return 'fingerprint'
    if c.startswith('emb_'):
        return 'learned embedding'
    if c.startswith('tp_'):
        return 'polymer topology'
    if c.startswith('true_'):
        return 'observed partner'
    if c.startswith('ph_'):
        return 'physics term'
    if c.startswith('grp_'):
        return 'functional group'
    if c.startswith('po_'):
        return 'custom descriptor'
    return 'rdkit descriptor'


def _explain():
    out = {}
    models = REF_MODELS or {}
    if HAVE_SHAP and models:
        for tt in TARGET_TYPES:
            if tt not in models:
                continue
            try:
                rows, _ = prop_folds(tt)
                X = drop_leaky(train_features.iloc[rows].reset_index(drop=True), tt)
                if len(X) > 600:
                    X = X.sample(600, random_state=SEED)
                sv = shap.TreeExplainer(models[tt]).shap_values(X)
                imp = np.abs(sv).mean(0)
                fam = {}
                for c, v in zip(X.columns, imp):
                    fam[_family(c)] = fam.get(_family(c), 0.0) + float(v)
                tot = sum(fam.values()) or 1.0
                fam = {k: round(100 * v / tot, 1) for k, v in
                       sorted(fam.items(), key=lambda kv: -kv[1])}
                top = [(str(c), round(float(v), 5)) for c, v in
                       sorted(zip(X.columns, imp), key=lambda kv: -kv[1])[:15]]
                out[tt] = dict(families_pct=fam, top_features=top)
                log.metric(f'  [{tt}] families: ' +
                           '  '.join(f'{k} {v}%' for k, v in list(fam.items())[:5]))
                log.info(f'         top: ' + ', '.join(c for c, _ in top[:8]))
            except Exception as e:
                log.warn(f'  SHAP failed for {tt}: {e}')
    else:
        log.warn('shap unavailable or no retained model - skipping SHAP')

    # ---- token saliency from the fine-tuned encoder ----
    sal = {}
    try:
        tagged = [f'ft/f{f}' for f in range(N_FOLDS) if fhas(f'ft/f{f}')]
        if tagged and pretrained.get('state') is not None:
            st = fload(tagged[0])
            model = PolymerRegressor(_encoder_from(pretrained['state']), PRE_CFG['d_model']).to(DEVICE)
            model.load_state_dict(st['best_state'] or st['model'])
            model.eval()
            for tt in TARGET_TYPES:
                m = (test_df.target_type == tt).values
                if not m.any():
                    continue
                smi = test_df.nsmiles.values[np.where(m)[0][0]]
                ids = torch.as_tensor(np.array([encode(smi, VOCAB)]), device=DEVICE)
                e = model.enc.tok(ids).detach().requires_grad_(True)
                pos = torch.arange(ids.size(1), device=DEVICE).unsqueeze(0)
                h = model.enc.drop(model.enc.norm(e + model.enc.pos(pos)))
                h = model.enc.enc(h, src_key_padding_mask=ids.eq(PAD))
                msk = (~ids.eq(PAD)).unsqueeze(-1).float()
                z = (h * msk).sum(1) / msk.sum(1).clamp(min=1.0)
                model.heads[TASK_MAP[tt]](z).squeeze().backward()
                g = e.grad.abs().sum(-1).squeeze(0).detach().cpu().numpy()
                toks = [INV_VOCAB.get(int(i), '?') for i in ids[0].cpu().numpy()]
                keep = [(t, float(v)) for t, v in zip(toks, g) if t not in ('<pad>',)]
                keep.sort(key=lambda kv: -kv[1])
                sal[tt] = dict(smiles=smi, top_tokens=[t for t, _ in keep[:10]])
                log.info(f'  [{tt}] saliency top tokens: {" ".join(t for t,_ in keep[:10])}')
            del model; gc.collect()
    except Exception as e:
        log.warn(f'  token saliency failed: {e}')

    out['_saliency'] = sal
    out['_physics'] = dict(true=PHYSR['true'], predicted=PHYSR['pred'])
    out['_stack_weights'] = STACK['weights']
    return out


EXPLAIN = run_stage('17_explain', _explain)

## 16. Submission

In [ ]:
log.set_stage('18_submission')
log.header('SUBMISSION')

for tt in TARGET_TYPES:
    m = (test_df.target_type == tt).values
    v = train_df.loc[train_df.target_type == tt, 'target']
    if not m.any() or v.empty:
        continue
    lo, hi = v.min(), v.max(); pad = 0.05 * (hi - lo)
    n_clip = int(((final[m] < lo - pad) | (final[m] > hi + pad)).sum())
    final[m] = np.clip(final[m], lo - pad, hi + pad)
    if n_clip:
        log.info(f'  {tt}: clipped {n_clip} prediction(s) to the observed range')

bad = ~np.isfinite(final)
if bad.any():
    log.warn(f'{bad.sum()} non-finite predictions -> LightGBM fallback')
    final[bad] = tree_test['lgbm'][bad]

sub = pd.DataFrame({'id': test_df.id.values, 'target': final})
assert len(sub) == len(test_df), 'row count changed'
assert sub.target.notna().all() and np.isfinite(sub.target.values).all(), 'non-finite prediction'
assert sub.id.nunique() == len(sub), 'duplicate id'
sub.to_csv(os.path.join(WORK_DIR, 'submission.csv'), index=False)
log.ok(f'submission.csv written {sub.shape}')
print(sub.groupby(test_df.target_type.values).target.agg(['count', 'min', 'mean', 'max']).round(3))

report = dict(
    generated=datetime.now().isoformat(),
    runtime_minutes=round(elapsed() / 60, 1),
    invariance_selftest=inv_selftest,
    invariance_audit=INV_AUDIT,
    grouped_oof_r2=dict(stack=stack_r2, mean=float(np.mean(list(stack_r2.values())))),
    base_model_r2={n: (tree_r2[n] if n in GBDT_KINDS else
                       (NNR['r2'] if n == 'nn' else (FT['r2'] if n == 'ft' else CNNR['r2'])))
                   for n in BASE_NAMES},
    stack_weights=STACK['weights'],
    physics=dict(true=PHYSR['true'], predicted=PHYSR['pred']),
    explainability={k: v for k, v in EXPLAIN.items() if not k.startswith('_')},
    saliency=EXPLAIN.get('_saliency', {}),
    pretraining=dict(steps=pretrained.get('steps'), final_loss=(pretrained.get('hist') or [{}])[-1]),
)
with open(os.path.join(WORK_DIR, 'report.json'), 'w') as h:
    json.dump(report, h, indent=1, default=str)

log.header('RUN COMPLETE')
log.metric(f'grouped-OOF mean R2 = {np.mean(list(stack_r2.values())):.4f}')
log.metric(f'total runtime      = {elapsed()/60:.1f} min')
log.ok('artifacts: submission.csv, report.json, run.log, events.jsonl, ckpt/')
sub.head()